# Llama-3-8B Full Generation: Jungle vs MagicPIG

**Purpose:** Complete end-to-end implementation and comparison of Jungle (forest-based SNIS) vs MagicPIG (fixed-depth LSH) for text generation with Llama-3-8B-Instruct.

**What it does:**
- Implements full LSH-based sparse attention server with both methods
- Replaces standard attention in Llama-3-8B with sparse attention mechanism
- Performs text generation with detailed logging of:
  - Tokens retrieved (local vs sparse)
  - Tree depths used (for Jungle)
  - Generation speed and quality
- Compares Jungle's adaptive depth selection vs MagicPIG's fixed depth

**Key components:**
- `LSHSparseAttnServer`: Main attention server with configurable Jungle/MagicPIG modes
- `jungle_snis_transform`: Depth-aware importance sampling correction
- `magicpig_transform`: Fixed-depth importance sampling correction

**Use case:** Demonstrates practical decoding-time speedup with sparse attention while maintaining generation quality.

In [1]:
import os
# os.environ["HF_TOKEN"] = "your_huggingface_token_here"  # Set your HF token or use `huggingface-cli login`

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
import math
import collections
import time
import gc
import numpy as np

# ==========================================
# Part 1: Utility Functions
# ==========================================

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
    q_f32 = q.float()
    cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
    sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
    q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
    return q_embed.to(q.dtype)

def manual_rmsnorm(hidden_states, weight, variance_epsilon):
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)
    variance = hidden_states.pow(2).mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return weight * hidden_states.to(input_dtype)

def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = cumulative_probs > top_p
    mask[:, :, 1:] = mask[:, :, :-1].clone()
    mask[:, :, 0] = False
    sorted_probs.masked_fill_(mask, 0.0)
    sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
    sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
    final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
    return final_indices.squeeze(-1)

# ==========================================
# Part 2: LSH Implementation (Legacy MagicPIG)
# ==========================================

class LSH:
    def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
        self.K = K
        self.L = L
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.batch_size = batch_size
        self.max_length = max_length
        self.num_attention_groups = num_heads // num_kv_heads
        self.device = device
        self.tables = []
        for _ in range(num_layers):
            req_tables = []
            for _ in range(batch_size):
                head_tables = []
                for _ in range(num_kv_heads):
                    l_tables = [collections.defaultdict(list) for _ in range(L)]
                    head_tables.append(l_tables)
                req_tables.append(head_tables)
            self.tables.append(req_tables)

    def clear(self):
        for layer_idx in range(self.num_layers):
            for req_id in range(self.batch_size):
                for head_idx in range(self.num_kv_heads):
                    for l in range(self.L):
                        self.tables[layer_idx][req_id][head_idx][l].clear()

    def fill(self, layer_id, request_id, hash_codes, indices):
        hc = hash_codes.cpu()
        idx = indices.cpu()
        num_kv, L, seq_len = hc.shape
        for h in range(num_kv):
            for l in range(L):
                table_dict = self.tables[layer_id][request_id][h][l]
                current_hashes = hc[h, l].tolist()
                current_indices = idx.tolist()
                for i, val in enumerate(current_hashes):
                    table_dict[val].append(current_indices[i])

    def batch_retrieve(self, layer_id, query_hash_codes):
        B_H, L = query_hash_codes.shape
        query_hash_codes = query_hash_codes.cpu()
        results = []
        for i in range(B_H):
            req_id = i // self.num_heads
            local_head_id = i % self.num_heads
            kv_head_id = local_head_id // self.num_attention_groups
            counts = collections.defaultdict(int)
            for l in range(L):
                val = query_hash_codes[i, l].item()
                bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
                for idx in bucket:
                    counts[idx] += 1
            candidates = [idx for idx, count in counts.items() if count >= 2]
            if not candidates:
                t_cand = torch.empty(0, dtype=torch.long, device=self.device)
            else:
                t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
            results.append(t_cand)
        return results

# ==========================================
# Part 3: Sparse Attention Math (SNIS Kernels)
# ==========================================

def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
    if k.shape[0] == 0:
        return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
    score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
    q_norm_f = q.float().norm(p=2)
    k_norm_f = k_norm.float()
    denom = q_norm_f * k_norm_f
    cos_theta = score_f / (denom + 1e-6)
    cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
    theta = torch.acos(cos_theta)
    prob = 1.0 - theta / math.pi
    p = prob.pow(K)
    q_prob = 1.0 - p
    w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
    log_w = torch.log(w + 1e-4)
    if is_exact is not None:
        log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
    score_f = score_f / math.sqrt(head_dim) - log_w
    attn_probs = torch.softmax(score_f, dim=0)
    return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
    if k.shape[0] == 0:
        return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
    score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
    q_norm_f = q.float().norm(p=2)
    k_norm_f = k_norm.float()
    denom = q_norm_f * k_norm_f
    cos_theta = score_f / (denom + 1e-6)
    cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
    theta = torch.acos(cos_theta)
    p_base = 1.0 - theta / math.pi

    # 1. Calculate collision prob at the realized depth
    p_collision = p_base.pow(retrieval_depths.float())

    # 2. Probability of at least one collision across L trees
    w = 1.0 - (1.0 - p_collision).pow(L)

    # --- FIX: Numerical Stability ---
    # Clamp probability to avoid log(0) without adding a large bias (1e-4)
    log_w = torch.log(torch.clamp(w, min=1e-10))
    # --------------------------------

    if is_exact is not None:
        log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

    score_f = score_f / math.sqrt(head_dim) - log_w
    attn_probs = torch.softmax(score_f, dim=0)
    return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# ==========================================
# Part 4: Logging Infrastructure
# ==========================================

class AttentionLogger:
    def __init__(self):
        self.reset()

    def reset(self):
        self.stats = collections.defaultdict(list)

    def log(self, key, value):
        self.stats[key].append(value)

    def summary(self, method_name):
        print("\n" + "="*60)
        print(f"   SPARSE ATTENTION SUMMARY: {method_name.upper()}")
        print("="*60)

        # General Stats
        n_steps = len(self.stats.get("total_tokens", []))
        if n_steps == 0:
            print("No data collected.")
            return

        total_toks = np.mean(self.stats["total_tokens"])
        sink = np.mean(self.stats["sink_tokens"])
        local = np.mean(self.stats["local_tokens"])
        sparse = np.mean(self.stats["sparse_tokens"])

        print(f"Average Sequence Context: {total_toks:.1f} tokens")
        print("-" * 40)
        print(f"{'Token Type':<20} | {'Avg Count':<10} | {'% of Total':<10}")
        print("-" * 40)
        print(f"{'Sink (Exact)':<20} | {sink:<10.1f} | {sink/total_toks*100:<10.1f}%")
        print(f"{'Local (Exact)':<20} | {local:<10.1f} | {local/total_toks*100:<10.1f}%")
        print(f"{'Retrieved (Sparse)':<20} | {sparse:<10.1f} | {sparse/total_toks*100:<10.1f}%")
        print(f"{'Ignored':<20} | {total_toks - sink - local - sparse:<10.1f} | {(total_toks - sink - local - sparse)/total_toks*100:<10.1f}%")
        print("-" * 40)

        # Method Specifics
        if method_name == "Jungle":
            avg_depth = np.mean(self.stats["avg_depth"])
            max_depth = np.max(self.stats["max_depth"])
            print(f"\n🌲 Jungle Specifics:")
            print(f"  - Average Tree Depth Used: {avg_depth:.2f}")
            print(f"  - Max Depth Reached:       {max_depth:.2f}")
        else:
            print(f"\n🐷 MagicPIG Specifics:")
            print(f"  - (Standard LSH Statistics not fully instrumented in this lightweight demo)")

        print("="*60 + "\n")

class LSHSparseAttnServer:
    def __init__(self, config, K=10, L=150, batch_size=1,
                 num_sink_tokens=4, num_local_tokens=64,
                 max_length=8192, dense_layers=[0, 16, 32],
                 device='cuda:0', dtype=torch.bfloat16, verbose=False,
                 use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

        self.config = config
        self.K = K
        self.L = L
        self.batch_size = batch_size
        self.num_sink_tokens = num_sink_tokens
        self.num_local_tokens = num_local_tokens
        self.dense_layers = set(dense_layers)
        self.device = device
        self.dtype = dtype
        self.verbose = verbose
        self.use_jungle = use_jungle

        # Logging
        self.logger = AttentionLogger()
        self.log_interval = 2
        self.logging_layer = 15

        # Jungle Params
        self.jg_budget = jg_budget
        self.jg_K_max = jg_K_max
        self.jg_L = jg_L
        self.jg_min_depth = jg_min_depth

        self.num_layers = config.num_hidden_layers
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.hidden_size // self.num_heads
        self.num_attention_groups = self.num_heads // self.num_kv_heads

        self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
        self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
        self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
        self.current_len = [0] * batch_size

        self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
        self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
        self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

        if self.use_jungle:
            print(f"🌲 Jungle Attention Enabled (Proper SNIS Mode)")
            self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
            self.jg_hash_cache = collections.defaultdict(dict)
        else:
            print(f"🐷 MagicPIG Attention Enabled")

        self.sparse_boundaries = {}

    def clear(self):
        self.lsh.clear()
        self.logger.reset()
        self.step_counter = 0
        for i in range(self.batch_size):
            self.current_len[i] = 0
        for l in range(self.num_layers):
            self.k_cache[l].zero_()
            self.v_cache[l].zero_()
            self.avg_k_cache[l].zero_()
        self.sparse_boundaries = {}
        if self.use_jungle:
            self.jg_hash_cache.clear()

    def step(self):
        self.step_counter += 1
        for i in range(self.batch_size):
            self.current_len[i] += 1

    def print_summary(self):
        method = "Jungle" if self.use_jungle else "MagicPIG"
        self.logger.summary(method)

    def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
        seq_len = key_states.shape[0]
        end_pos = start_pos + seq_len
        self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
        self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
        if end_pos > self.current_len[request_id]:
            self.current_len[request_id] = end_pos

        if layer_idx not in self.dense_layers:
            idx_start = max(start_pos, self.num_sink_tokens)
            idx_end = end_pos - self.num_local_tokens
            self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

            if idx_end > idx_start:
                keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
                avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
                self.avg_k_cache[layer_idx][request_id] = avg_k
                centered_keys = keys_to_index - avg_k

                if self.use_jungle:
                    jg_proj = torch.matmul(centered_keys, self.jg_projs)
                    jg_bits = (jg_proj > 0).float()
                    n_sparse = jg_bits.shape[1]
                    jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
                    if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
                    self.jg_hash_cache[layer_idx][request_id] = jg_bits
                else:
                    projected = torch.matmul(centered_keys, self.hash_func)
                    bits = (projected > 0).float()
                    bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
                    buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
                    indices = torch.arange(idx_start, idx_end, device=self.device)
                    self.lsh.fill(layer_idx, request_id, buckets, indices)

    def decode(self, query_states, key_states, value_states, layer_idx):
        bsz, n_heads, q_len, dim = query_states.shape
        for req_id in range(bsz):
            curr_len = self.current_len[req_id]
            self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
            self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

        hidden_states_list = []

        # Init counters
        log_sink = 0
        log_local = 0
        log_sparse = 0
        log_depths = []

        for req_id in range(bsz):
            q_heads = query_states[req_id, :, 0, :]
            curr_len = self.current_len[req_id]

            if layer_idx in self.dense_layers:
                k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
                v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
                k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
                v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
                scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
                attn = torch.softmax(scores, dim=-1)
                out = torch.matmul(attn, v.float()).squeeze(1)
                hidden_states_list.append(out.to(self.dtype))
            else:
                head_outputs = []
                norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

                if self.use_jungle:
                    jg_q_proj = torch.matmul(norm_q, self.jg_projs)
                    jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
                else:
                    projected = torch.matmul(norm_q, self.hash_func)
                    bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
                    q_buckets = torch.matmul(bits, self.binary_pack).long()
                    idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

                for h in range(self.num_heads):
                    kv_head = h // self.num_attention_groups
                    sparse_boundary = self.sparse_boundaries.get(req_id, 0)

                    sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
                    local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

                    sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
                    retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

                    if self.use_jungle:
                        if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
                            k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
                            q_bits_h = jg_q_bits[h]
                            match = (k_bits == q_bits_h.unsqueeze(0)).int()
                            depths = match.cumprod(dim=-1).sum(dim=-1)
                            max_d, _ = depths.max(dim=-1)

                            N_sparse = max_d.shape[0]
                            budget = int(N_sparse * self.jg_budget)
                            if budget > 0:
                                sorted_d, sorted_idx = torch.sort(max_d, descending=True)
                                valid_mask = sorted_d >= self.jg_min_depth
                                valid_idx = sorted_idx[valid_mask]
                                take = min(budget, valid_idx.numel())
                                if take > 0:
                                    sparse_indices = valid_idx[:take] + self.num_sink_tokens
                                    retrieved_depths = sorted_d[:take].float()
                                    if h == 0 and req_id == 0:
                                        log_depths.extend(retrieved_depths.tolist())
                    else:
                        raw_indices = idx_list[h]
                        if raw_indices.numel() > 0:
                            sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

                    if h == 0 and req_id == 0:
                        log_sink = sink_indices.numel()
                        log_local = local_indices.numel()
                        log_sparse = sparse_indices.numel()

                    full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

                    # --- CRITICAL FIX START (Probability Calculation) ---
                    depth_map = torch.zeros(full_indices.shape[0], device=self.device)

                    if self.use_jungle and sparse_indices.numel() > 0:
                        # 1. Identify the effective threshold used for this batch
                        effective_threshold = retrieved_depths.min().item()

                        # 2. Assign this threshold to ALL retrieved keys
                        # We do NOT use the individual depths. We use the threshold that qualified them.
                        sparse_set = set(sparse_indices.tolist())

                        # If index is sparse, it gets threshold depth.
                        # If it is exact (sink/local), it gets 0.0 (which results in p=1.0, w=1.0, log_w=0)
                        depth_list = [effective_threshold if idx.item() in sparse_set else 0.0 for idx in full_indices]
                        depth_map = torch.tensor(depth_list, device=self.device)
                    # --- CRITICAL FIX END ---

                    k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
                    v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
                    avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
                    k_sel_centered = k_sel - avg_k
                    k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
                    is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

                    if self.use_jungle:
                        out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
                    else:
                        out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)
                    head_outputs.append(out_h)
                hidden_states_list.append(torch.cat(head_outputs, dim=0))

        if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
            self.logger.log("total_tokens", self.current_len[0])
            self.logger.log("sink_tokens", log_sink)
            self.logger.log("local_tokens", log_local)
            self.logger.log("sparse_tokens", log_sparse)
            if self.use_jungle and log_depths:
                self.logger.log("avg_depth", np.mean(log_depths))
                self.logger.log("max_depth", np.max(log_depths))

        return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
# ==========================================
# Part 6: Model Wrappers
# ==========================================

class LLMLayer:
    def __init__(self, layer_idx, hf_layer, device):
        self.layer_idx = layer_idx
        self.device = device
        self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
        self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
        self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
        self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
        self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
        self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
        self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
        self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
        self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
        self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
        self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

    def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
        residual = hidden_states
        hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
        bsz, q_len, _ = hidden_states.shape
        q = F.linear(hidden_states, self.wq)
        k = F.linear(hidden_states, self.wk)
        v = F.linear(hidden_states, self.wv)
        n_heads = attn_server.num_heads
        n_kv_heads = attn_server.num_kv_heads
        head_dim = attn_server.head_dim
        q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
        k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
        v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
        q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
        k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

        if is_prefill:
            for i in range(bsz):
                attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
            k_rep = repeat_kv(k, n_heads // n_kv_heads)
            v_rep = repeat_kv(v, n_heads // n_kv_heads)
            attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
            attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
        else:
            attn_output = attn_server.decode(q, k, v, self.layer_idx)

        hidden_states = F.linear(attn_output, self.wo)
        hidden_states = residual + hidden_states
        residual = hidden_states
        hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
        up = F.linear(hidden_states, self.up_proj)
        gate = F.linear(hidden_states, self.gate_proj)
        down = F.linear(F.silu(gate) * up, self.down_proj)
        hidden_states = residual + down
        return hidden_states

class LLM:
    def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
        self.device = device
        self.config = LlamaConfig.from_pretrained(model_name)
        self.max_length = max_length
        print(f"Loading Model: {model_name}...")
        hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
        self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
        self.lm_head = hf_model.lm_head.weight.detach().to(device)
        self.norm_weight = hf_model.model.norm.weight.detach().to(device)
        self.norm_eps = hf_model.model.norm.variance_epsilon
        self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
        t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.cos_cache = emb.cos().to(torch.bfloat16)
        self.sin_cache = emb.sin().to(torch.bfloat16)
        self.layers = []
        for idx, layer in enumerate(hf_model.model.layers):
            self.layers.append(LLMLayer(idx, layer, device))
            hf_model.model.layers[idx] = None
            gc.collect()
        self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

    def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
        self.attn_server.clear()
        self.attn_server.verbose = verbose
        bsz, seq_len = input_ids.shape
        position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
        print("Prefilling...")
        t0 = time.time()
        hidden_states = F.embedding(input_ids, self.embed_tokens)
        for layer in self.layers:
            hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
        generated = []
        curr_pos = seq_len
        logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
        next_token = topp_temperature_decode(logits, temperature)
        generated.append(next_token.item())
        t1 = time.time()
        print(f"Prefill done in {t1-t0:.2f}s")
        print("Generating...")
        for i in range(max_tokens):
            if verbose: print(f"--- Step {i} ---")
            input_ids = next_token
            position_ids = torch.tensor([[curr_pos]], device=self.device)
            hidden_states = F.embedding(input_ids, self.embed_tokens)
            for layer in self.layers:
                hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
            self.attn_server.step()
            logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
            next_token = topp_temperature_decode(logits, temperature)
            generated.append(next_token.item())
            curr_pos += 1
            if next_token.item() in [128001, 128009]:
                break

        # PRINT SUMMARY AT THE END
        self.attn_server.print_summary()
        return generated

if __name__ == "__main__":
    MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

    prompt = "Answer the question and then explain. In the rapidly evolving field of elementary mathematics, teachers such as David are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is 50. Ok now that the math is done, let's tell a story but start with the name of the teacher we mentioned. Once upon a time, "
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

    # # 1. RUN MAGICPIG
    # print("\n\n" + "#"*40)
    # print("RUNNING MAGICPIG (LSH)")
    # print("#"*40)
    # llm_mp = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
    # out_mp = llm_mp.generate(input_ids, max_tokens=20, verbose=False)
    # print(f"Generated (MagicPIG): {tokenizer.decode(out_mp)}")

    # 2. RUN JUNGLE
    print("\n\n" + "#"*40)
    print("RUNNING JUNGLE (Proper SNIS)")
    print("#"*40)
    llm_jg = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)
    out_jg = llm_jg.generate(input_ids, max_tokens=20, verbose=False)
    print(f"Generated (Jungle): {tokenizer.decode(out_jg)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!




########################################
RUNNING JUNGLE (Proper SNIS)
########################################
Loading Model: meta-llama/Meta-Llama-3-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌲 Jungle Attention Enabled (Proper SNIS Mode)
Prefilling...
Prefill done in 0.52s
Generating...

   SPARSE ATTENTION SUMMARY: JUNGLE
Average Sequence Context: 200.0 tokens
----------------------------------------
Token Type           | Avg Count  | % of Total
----------------------------------------
Sink (Exact)         | 4.0        | 2.0       %
Local (Exact)        | 73.0       | 36.5      %
Retrieved (Sparse)   | 6.0        | 3.0       %
Ignored              | 117.0      | 58.5      %
----------------------------------------

🌲 Jungle Specifics:
  - Average Tree Depth Used: 12.80
  - Max Depth Reached:       17.00

Generated (Jungle): 20 years ago, there was a teacher named David who was passionate about teaching mathematics to his students. He


In [3]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc
# import numpy as np

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#     mask = cumulative_probs > top_p
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False
#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation (Legacy MagicPIG)
# # ==========================================

# class LSH:
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()
#         results = []
#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups
#             counts = collections.defaultdict(int)
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1
#             candidates = [idx for idx, count in counts.items() if count >= 2]
#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
#             results.append(t_cand)
#         return results

# # ==========================================
# # Part 3: Sparse Attention Math (SNIS Kernels)
# # ==========================================

# def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     p_base = 1.0 - theta / math.pi

#     # 1. Calculate collision prob at the realized depth
#     p_collision = p_base.pow(retrieval_depths.float())

#     # 2. Probability of at least one collision across L trees
#     w = 1.0 - (1.0 - p_collision).pow(L)

#     # --- FIX: Numerical Stability ---
#     # Clamp probability to avoid log(0) without adding a large bias (1e-4)
#     log_w = torch.log(torch.clamp(w, min=1e-10))
#     # --------------------------------

#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# # ==========================================
# # Part 4: Logging Infrastructure
# # ==========================================

# class AttentionLogger:
#     def __init__(self):
#         self.reset()

#     def reset(self):
#         self.stats = collections.defaultdict(list)

#     def log(self, key, value):
#         self.stats[key].append(value)

#     def summary(self, method_name):
#         print("\n" + "="*60)
#         print(f"   SPARSE ATTENTION SUMMARY: {method_name.upper()}")
#         print("="*60)

#         # General Stats
#         n_steps = len(self.stats.get("total_tokens", []))
#         if n_steps == 0:
#             print("No data collected.")
#             return

#         total_toks = np.mean(self.stats["total_tokens"])
#         sink = np.mean(self.stats["sink_tokens"])
#         local = np.mean(self.stats["local_tokens"])
#         sparse = np.mean(self.stats["sparse_tokens"])

#         print(f"Average Sequence Context: {total_toks:.1f} tokens")
#         print("-" * 40)
#         print(f"{'Token Type':<20} | {'Avg Count':<10} | {'% of Total':<10}")
#         print("-" * 40)
#         print(f"{'Sink (Exact)':<20} | {sink:<10.1f} | {sink/total_toks*100:<10.1f}%")
#         print(f"{'Local (Exact)':<20} | {local:<10.1f} | {local/total_toks*100:<10.1f}%")
#         print(f"{'Retrieved (Sparse)':<20} | {sparse:<10.1f} | {sparse/total_toks*100:<10.1f}%")
#         print(f"{'Ignored':<20} | {total_toks - sink - local - sparse:<10.1f} | {(total_toks - sink - local - sparse)/total_toks*100:<10.1f}%")
#         print("-" * 40)

#         # Method Specifics
#         if method_name == "Jungle":
#             avg_depth = np.mean(self.stats["avg_depth"])
#             max_depth = np.max(self.stats["max_depth"])
#             print(f"\n🌲 Jungle Specifics:")
#             print(f"  - Average Tree Depth Used: {avg_depth:.2f}")
#             print(f"  - Max Depth Reached:       {max_depth:.2f}")
#         else:
#             print(f"\n🐷 MagicPIG Specifics:")
#             print(f"  - (Standard LSH Statistics not fully instrumented in this lightweight demo)")

#         print("="*60 + "\n")

# class LSHSparseAttnServer:
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose
#         self.use_jungle = use_jungle

#         # Logging
#         self.logger = AttentionLogger()
#         self.log_interval = 2
#         self.logging_layer = 15

#         # Jungle Params
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.current_len = [0] * batch_size

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (Proper SNIS Mode)")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             self.jg_hash_cache = collections.defaultdict(dict)
#         else:
#             print(f"🐷 MagicPIG Attention Enabled")

#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         self.logger.reset()
#         self.step_counter = 0
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         self.step_counter += 1
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def print_summary(self):
#         method = "Jungle" if self.use_jungle else "MagicPIG"
#         self.logger.summary(method)

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         if layer_idx not in self.dense_layers:
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k
#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float()
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         # Init counters
#         log_sink = 0
#         log_local = 0
#         log_sparse = 0
#         log_depths = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))
#             else:
#                 head_outputs = []
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 if self.use_jungle:
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
#                     local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1)
#                             max_d, _ = depths.max(dim=-1)

#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())
#                                 if take > 0:
#                                     sparse_indices = valid_idx[:take] + self.num_sink_tokens
#                                     retrieved_depths = sorted_d[:take].float()
#                                     if h == 0 and req_id == 0:
#                                         log_depths.extend(retrieved_depths.tolist())
#                     else:
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

#                     if h == 0 and req_id == 0:
#                         log_sink = sink_indices.numel()
#                         log_local = local_indices.numel()
#                         log_sparse = sparse_indices.numel()

#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     # --- CRITICAL FIX START (Probability Calculation) ---
#                     depth_map = torch.zeros(full_indices.shape[0], device=self.device)

#                     if self.use_jungle and sparse_indices.numel() > 0:
#                         # 1. Identify the effective threshold used for this batch
#                         effective_threshold = retrieved_depths.min().item()

#                         # 2. Assign this threshold to ALL retrieved keys
#                         # We do NOT use the individual depths. We use the threshold that qualified them.
#                         sparse_set = set(sparse_indices.tolist())

#                         # If index is sparse, it gets threshold depth.
#                         # If it is exact (sink/local), it gets 0.0 (which results in p=1.0, w=1.0, log_w=0)
#                         depth_list = [effective_threshold if idx.item() in sparse_set else 0.0 for idx in full_indices]
#                         depth_map = torch.tensor(depth_list, device=self.device)
#                     # --- CRITICAL FIX END ---

#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

#                     if self.use_jungle:
#                         out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
#                     else:
#                         out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)
#                     head_outputs.append(out_h)
#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
#             self.logger.log("total_tokens", self.current_len[0])
#             self.logger.log("sink_tokens", log_sink)
#             self.logger.log("local_tokens", log_local)
#             self.logger.log("sparse_tokens", log_sparse)
#             if self.use_jungle and log_depths:
#                 self.logger.log("avg_depth", np.mean(log_depths))
#                 self.logger.log("max_depth", np.max(log_depths))

#         return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
# # ==========================================
# # Part 6: Model Wrappers
# # ==========================================

# class LLMLayer:
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device
#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)
#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim
#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             for i in range(bsz):
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down
#         return hidden_states

# class LLM:
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length
#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon
#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)
#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()
#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.clear()
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
#         print("Prefilling...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
#         generated = []
#         curr_pos = seq_len
#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")
#         print("Generating...")
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token
#             position_ids = torch.tensor([[curr_pos]], device=self.device)
#             hidden_states = F.embedding(input_ids, self.embed_tokens)
#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
#             self.attn_server.step()
#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1
#             if next_token.item() in [128001, 128009]:
#                 break

#         # PRINT SUMMARY AT THE END
#         self.attn_server.print_summary()
#         return generated

# if __name__ == "__main__":
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     prompt = "Answer the question and then explain. In the rapidly evolving field of elementary mathematics, teachers such as David are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is 50. Ok now that the math is done, let's tell a story but start with the name of the teacher we mentioned. Once upon a time, "
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#     # # 1. RUN MAGICPIG
#     # print("\n\n" + "#"*40)
#     # print("RUNNING MAGICPIG (LSH)")
#     # print("#"*40)
#     # llm_mp = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
#     # out_mp = llm_mp.generate(input_ids, max_tokens=20, verbose=False)
#     # print(f"Generated (MagicPIG): {tokenizer.decode(out_mp)}")

#     # 2. RUN JUNGLE
#     print("\n\n" + "#"*40)
#     print("RUNNING JUNGLE (Proper SNIS)")
#     print("#"*40)
#     llm_jg = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)
#     out_jg = llm_jg.generate(input_ids, max_tokens=50, verbose=False)
#     print(f"Generated (Jungle): {tokenizer.decode(out_jg)}")

In [4]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc
# import numpy as np

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#     mask = cumulative_probs > top_p
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False
#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation (Legacy MagicPIG)
# # ==========================================

# class LSH:
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()
#         results = []
#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups
#             counts = collections.defaultdict(int)
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1
#             candidates = [idx for idx, count in counts.items() if count >= 2]
#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
#             results.append(t_cand)
#         return results

# # ==========================================
# # Part 3: Sparse Attention Math (SNIS Kernels)
# # ==========================================

# def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     p_base = 1.0 - theta / math.pi
#     p_collision = p_base.pow(retrieval_depths.float())
#     w = 1.0 - (1.0 - p_collision).pow(L)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# # ==========================================
# # Part 4: Logging Infrastructure
# # ==========================================

# class AttentionLogger:
#     def __init__(self):
#         self.reset()

#     def reset(self):
#         self.stats = collections.defaultdict(list)

#     def log(self, key, value):
#         self.stats[key].append(value)

#     def summary(self, method_name):
#         print("\n" + "="*60)
#         print(f"   SPARSE ATTENTION SUMMARY: {method_name.upper()}")
#         print("="*60)

#         # General Stats
#         n_steps = len(self.stats.get("total_tokens", []))
#         if n_steps == 0:
#             print("No data collected.")
#             return

#         total_toks = np.mean(self.stats["total_tokens"])
#         sink = np.mean(self.stats["sink_tokens"])
#         local = np.mean(self.stats["local_tokens"])
#         sparse = np.mean(self.stats["sparse_tokens"])

#         print(f"Average Sequence Context: {total_toks:.1f} tokens")
#         print("-" * 40)
#         print(f"{'Token Type':<20} | {'Avg Count':<10} | {'% of Total':<10}")
#         print("-" * 40)
#         print(f"{'Sink (Exact)':<20} | {sink:<10.1f} | {sink/total_toks*100:<10.1f}%")
#         print(f"{'Local (Exact)':<20} | {local:<10.1f} | {local/total_toks*100:<10.1f}%")
#         print(f"{'Retrieved (Sparse)':<20} | {sparse:<10.1f} | {sparse/total_toks*100:<10.1f}%")
#         print(f"{'Ignored':<20} | {total_toks - sink - local - sparse:<10.1f} | {(total_toks - sink - local - sparse)/total_toks*100:<10.1f}%")
#         print("-" * 40)

#         # Method Specifics
#         if method_name == "Jungle":
#             avg_depth = np.mean(self.stats["avg_depth"])
#             max_depth = np.max(self.stats["max_depth"])
#             print(f"\n🌲 Jungle Specifics:")
#             print(f"  - Average Tree Depth Used: {avg_depth:.2f}")
#             print(f"  - Max Depth Reached:       {max_depth:.2f}")
#         else:
#             print(f"\n🐷 MagicPIG Specifics:")
#             print(f"  - (Standard LSH Statistics not fully instrumented in this lightweight demo)")

#         print("="*60 + "\n")

# # ==========================================
# # Part 5: Attention Server (Corrected Logging)
# # ==========================================

# class LSHSparseAttnServer:
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose
#         self.use_jungle = use_jungle

#         # Logging
#         self.logger = AttentionLogger()
#         self.log_interval = 2 # Log frequently for this demo
#         self.step_counter = 0

#         # --- FIX: Log a layer that is actually Sparse! ---
#         # Layer 16 is dense, so we log Layer 15 instead.
#         self.logging_layer = 15

#         # Jungle Params
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.current_len = [0] * batch_size

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (Proper SNIS Mode)")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             self.jg_hash_cache = collections.defaultdict(dict)
#         else:
#             print(f"🐷 MagicPIG Attention Enabled")

#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         self.logger.reset()
#         self.step_counter = 0
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         self.step_counter += 1
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def print_summary(self):
#         method = "Jungle" if self.use_jungle else "MagicPIG"
#         self.logger.summary(method)

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         if layer_idx not in self.dense_layers:
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k
#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float()
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         # Init counters
#         log_sink = 0
#         log_local = 0
#         log_sparse = 0
#         log_depths = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 # Dense Layers (We don't log here anymore)
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))
#             else:
#                 # Sparse Layers (This logic runs for Layer 15)
#                 head_outputs = []
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 if self.use_jungle:
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
#                     local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1)
#                             max_d, _ = depths.max(dim=-1)

#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())
#                                 if take > 0:
#                                     sparse_indices = valid_idx[:take] + self.num_sink_tokens
#                                     retrieved_depths = sorted_d[:take].float()
#                                     if h == 0 and req_id == 0:
#                                         log_depths.extend(retrieved_depths.tolist())
#                     else:
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

#                     # Update Logging Counts
#                     if h == 0 and req_id == 0:
#                         log_sink = sink_indices.numel()
#                         log_local = local_indices.numel()
#                         log_sparse = sparse_indices.numel()

#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     depth_map = torch.zeros(full_indices.shape[0], device=self.device)
#                     if self.use_jungle and sparse_indices.numel() > 0:
#                         sparse_idx_to_depth = dict(zip(sparse_indices.tolist(), retrieved_depths.tolist()))
#                         depth_list = [sparse_idx_to_depth.get(idx.item(), 0.0) for idx in full_indices]
#                         depth_map = torch.tensor(depth_list, device=self.device)

#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

#                     if self.use_jungle:
#                         out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
#                     else:
#                         out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)
#                     head_outputs.append(out_h)
#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         # Commit logs (Use self.logging_layer instead of 16)
#         if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
#             self.logger.log("total_tokens", self.current_len[0])
#             self.logger.log("sink_tokens", log_sink)
#             self.logger.log("local_tokens", log_local)
#             self.logger.log("sparse_tokens", log_sparse)
#             if self.use_jungle and log_depths:
#                 self.logger.log("avg_depth", np.mean(log_depths))
#                 self.logger.log("max_depth", np.max(log_depths))

#         return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
# # ==========================================
# # Part 6: Model Wrappers
# # ==========================================

# class LLMLayer:
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device
#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)
#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim
#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             for i in range(bsz):
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down
#         return hidden_states

# class LLM:
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length
#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon
#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)
#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()
#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.clear()
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
#         print("Prefilling...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
#         generated = []
#         curr_pos = seq_len
#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")
#         print("Generating...")
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token
#             position_ids = torch.tensor([[curr_pos]], device=self.device)
#             hidden_states = F.embedding(input_ids, self.embed_tokens)
#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
#             self.attn_server.step()
#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1
#             if next_token.item() in [128001, 128009]:
#                 break

#         # PRINT SUMMARY AT THE END
#         self.attn_server.print_summary()
#         return generated

# if __name__ == "__main__":
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     prompt = "Answer the question and then explain. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is "
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#     # # 1. RUN MAGICPIG
#     # print("\n\n" + "#"*40)
#     # print("RUNNING MAGICPIG (LSH)")
#     # print("#"*40)
#     # llm_mp = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
#     # out_mp = llm_mp.generate(input_ids, max_tokens=20, verbose=False)
#     # print(f"Generated (MagicPIG): {tokenizer.decode(out_mp)}")

#     # 2. RUN JUNGLE
#     print("\n\n" + "#"*40)
#     print("RUNNING JUNGLE (Proper SNIS)")
#     print("#"*40)
#     llm_jg = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)
#     out_jg = llm_jg.generate(input_ids, max_tokens=50, verbose=False)
#     print(f"Generated (Jungle): {tokenizer.decode(out_jg)}")

In [5]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc
# import numpy as np

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#     mask = cumulative_probs > top_p
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False
#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation (Legacy MagicPIG)
# # ==========================================

# class LSH:
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()
#         results = []
#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups
#             counts = collections.defaultdict(int)
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1
#             candidates = [idx for idx, count in counts.items() if count >= 2]
#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
#             results.append(t_cand)
#         return results

# # ==========================================
# # Part 3: Sparse Attention Math (SNIS Kernels)
# # ==========================================

# def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     p_base = 1.0 - theta / math.pi
#     p_collision = p_base.pow(retrieval_depths.float())
#     w = 1.0 - (1.0 - p_collision).pow(L)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# # ==========================================
# # Part 4: Logging Infrastructure
# # ==========================================

# class AttentionLogger:
#     def __init__(self):
#         self.reset()

#     def reset(self):
#         self.stats = collections.defaultdict(list)

#     def log(self, key, value):
#         self.stats[key].append(value)

#     def summary(self, method_name):
#         print("\n" + "="*60)
#         print(f"   SPARSE ATTENTION SUMMARY: {method_name.upper()}")
#         print("="*60)

#         # General Stats
#         n_steps = len(self.stats.get("total_tokens", []))
#         if n_steps == 0:
#             print("No data collected.")
#             return

#         total_toks = np.mean(self.stats["total_tokens"])
#         sink = np.mean(self.stats["sink_tokens"])
#         local = np.mean(self.stats["local_tokens"])
#         sparse = np.mean(self.stats["sparse_tokens"])

#         print(f"Average Sequence Context: {total_toks:.1f} tokens")
#         print("-" * 40)
#         print(f"{'Token Type':<20} | {'Avg Count':<10} | {'% of Total':<10}")
#         print("-" * 40)
#         print(f"{'Sink (Exact)':<20} | {sink:<10.1f} | {sink/total_toks*100:<10.1f}%")
#         print(f"{'Local (Exact)':<20} | {local:<10.1f} | {local/total_toks*100:<10.1f}%")
#         print(f"{'Retrieved (Sparse)':<20} | {sparse:<10.1f} | {sparse/total_toks*100:<10.1f}%")
#         print(f"{'Ignored':<20} | {total_toks - sink - local - sparse:<10.1f} | {(total_toks - sink - local - sparse)/total_toks*100:<10.1f}%")
#         print("-" * 40)

#         # Method Specifics
#         if method_name == "Jungle":
#             avg_depth = np.mean(self.stats["avg_depth"])
#             max_depth = np.max(self.stats["max_depth"])
#             print(f"\n🌲 Jungle Specifics:")
#             print(f"  - Average Tree Depth Used: {avg_depth:.2f}")
#             print(f"  - Max Depth Reached:       {max_depth:.2f}")
#         else:
#             print(f"\n🐷 MagicPIG Specifics:")
#             print(f"  - (Standard LSH Statistics not fully instrumented in this lightweight demo)")

#         print("="*60 + "\n")

# # ==========================================
# # Part 5: Attention Server (Corrected Logging)
# # ==========================================

# class LSHSparseAttnServer:
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose
#         self.use_jungle = use_jungle

#         # Logging
#         self.logger = AttentionLogger()
#         self.log_interval = 2 # Log frequently for this demo
#         self.step_counter = 0

#         # --- FIX: Log a layer that is actually Sparse! ---
#         # Layer 16 is dense, so we log Layer 15 instead.
#         self.logging_layer = 15

#         # Jungle Params
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.current_len = [0] * batch_size

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (Proper SNIS Mode)")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             self.jg_hash_cache = collections.defaultdict(dict)
#         else:
#             print(f"🐷 MagicPIG Attention Enabled")

#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         self.logger.reset()
#         self.step_counter = 0
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         self.step_counter += 1
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def print_summary(self):
#         method = "Jungle" if self.use_jungle else "MagicPIG"
#         self.logger.summary(method)

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         if layer_idx not in self.dense_layers:
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k
#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float()
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         # Init counters
#         log_sink = 0
#         log_local = 0
#         log_sparse = 0
#         log_depths = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 # Dense Layers (We don't log here anymore)
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))
#             else:
#                 # Sparse Layers (This logic runs for Layer 15)
#                 head_outputs = []
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 if self.use_jungle:
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
#                     local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1)
#                             max_d, _ = depths.max(dim=-1)

#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())
#                                 if take > 0:
#                                     sparse_indices = valid_idx[:take] + self.num_sink_tokens
#                                     retrieved_depths = sorted_d[:take].float()
#                                     if h == 0 and req_id == 0:
#                                         log_depths.extend(retrieved_depths.tolist())
#                     else:
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

#                     # Update Logging Counts
#                     if h == 0 and req_id == 0:
#                         log_sink = sink_indices.numel()
#                         log_local = local_indices.numel()
#                         log_sparse = sparse_indices.numel()

#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     depth_map = torch.zeros(full_indices.shape[0], device=self.device)
#                     if self.use_jungle and sparse_indices.numel() > 0:
#                         sparse_idx_to_depth = dict(zip(sparse_indices.tolist(), retrieved_depths.tolist()))
#                         depth_list = [sparse_idx_to_depth.get(idx.item(), 0.0) for idx in full_indices]
#                         depth_map = torch.tensor(depth_list, device=self.device)

#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

#                     if self.use_jungle:
#                         out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
#                     else:
#                         out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)
#                     head_outputs.append(out_h)
#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         # Commit logs (Use self.logging_layer instead of 16)
#         if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
#             self.logger.log("total_tokens", self.current_len[0])
#             self.logger.log("sink_tokens", log_sink)
#             self.logger.log("local_tokens", log_local)
#             self.logger.log("sparse_tokens", log_sparse)
#             if self.use_jungle and log_depths:
#                 self.logger.log("avg_depth", np.mean(log_depths))
#                 self.logger.log("max_depth", np.max(log_depths))

#         return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
# # ==========================================
# # Part 6: Model Wrappers
# # ==========================================

# class LLMLayer:
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device
#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)
#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim
#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             for i in range(bsz):
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down
#         return hidden_states

# class LLM:
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length
#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon
#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)
#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()
#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.clear()
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
#         print("Prefilling...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
#         generated = []
#         curr_pos = seq_len
#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")
#         print("Generating...")
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token
#             position_ids = torch.tensor([[curr_pos]], device=self.device)
#             hidden_states = F.embedding(input_ids, self.embed_tokens)
#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
#             self.attn_server.step()
#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1
#             if next_token.item() in [128001, 128009]:
#                 break

#         # PRINT SUMMARY AT THE END
#         self.attn_server.print_summary()
#         return generated

# if __name__ == "__main__":
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     prompt = "Answer very concisely. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is "
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#     # # 1. RUN MAGICPIG
#     # print("\n\n" + "#"*40)
#     # print("RUNNING MAGICPIG (LSH)")
#     # print("#"*40)
#     # llm_mp = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
#     # out_mp = llm_mp.generate(input_ids, max_tokens=20, verbose=False)
#     # print(f"Generated (MagicPIG): {tokenizer.decode(out_mp)}")

#     # 2. RUN JUNGLE
#     print("\n\n" + "#"*40)
#     print("RUNNING JUNGLE (Proper SNIS)")
#     print("#"*40)
#     llm_jg = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)
#     out_jg = llm_jg.generate(input_ids, max_tokens=20, verbose=False)
#     print(f"Generated (Jungle): {tokenizer.decode(out_jg)}")

In [6]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc
# import numpy as np

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#     mask = cumulative_probs > top_p
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False
#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation (Legacy MagicPIG)
# # ==========================================

# class LSH:
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()
#         results = []
#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups
#             counts = collections.defaultdict(int)
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1
#             candidates = [idx for idx, count in counts.items() if count >= 2]
#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
#             results.append(t_cand)
#         return results

# # ==========================================
# # Part 3: Sparse Attention Math (SNIS Kernels)
# # ==========================================

# def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     p_base = 1.0 - theta / math.pi
#     p_collision = p_base.pow(retrieval_depths.float())
#     w = 1.0 - (1.0 - p_collision).pow(L)
#     log_w = torch.log(w + 1e-4)
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)
#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# # ==========================================
# # Part 4: Logging Infrastructure
# # ==========================================

# class AttentionLogger:
#     def __init__(self):
#         self.reset()

#     def reset(self):
#         self.stats = collections.defaultdict(list)

#     def log(self, key, value):
#         self.stats[key].append(value)

#     def summary(self, method_name):
#         print("\n" + "="*60)
#         print(f"   SPARSE ATTENTION SUMMARY: {method_name.upper()}")
#         print("="*60)

#         # General Stats
#         n_steps = len(self.stats.get("total_tokens", []))
#         if n_steps == 0:
#             print("No data collected.")
#             return

#         total_toks = np.mean(self.stats["total_tokens"])
#         sink = np.mean(self.stats["sink_tokens"])
#         local = np.mean(self.stats["local_tokens"])
#         sparse = np.mean(self.stats["sparse_tokens"])

#         print(f"Average Sequence Context: {total_toks:.1f} tokens")
#         print("-" * 40)
#         print(f"{'Token Type':<20} | {'Avg Count':<10} | {'% of Total':<10}")
#         print("-" * 40)
#         print(f"{'Sink (Exact)':<20} | {sink:<10.1f} | {sink/total_toks*100:<10.1f}%")
#         print(f"{'Local (Exact)':<20} | {local:<10.1f} | {local/total_toks*100:<10.1f}%")
#         print(f"{'Retrieved (Sparse)':<20} | {sparse:<10.1f} | {sparse/total_toks*100:<10.1f}%")
#         print(f"{'Ignored':<20} | {total_toks - sink - local - sparse:<10.1f} | {(total_toks - sink - local - sparse)/total_toks*100:<10.1f}%")
#         print("-" * 40)

#         # Method Specifics
#         if method_name == "Jungle":
#             avg_depth = np.mean(self.stats["avg_depth"])
#             max_depth = np.max(self.stats["max_depth"])
#             print(f"\n🌲 Jungle Specifics:")
#             print(f"  - Average Tree Depth Used: {avg_depth:.2f}")
#             print(f"  - Max Depth Reached:       {max_depth:.2f}")
#         else:
#             print(f"\n🐷 MagicPIG Specifics:")
#             print(f"  - (Standard LSH Statistics not fully instrumented in this lightweight demo)")

#         print("="*60 + "\n")

# # ==========================================
# # Part 5: Attention Server (Corrected Logging)
# # ==========================================

# class LSHSparseAttnServer:
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose
#         self.use_jungle = use_jungle

#         # Logging
#         self.logger = AttentionLogger()
#         self.log_interval = 2 # Log frequently for this demo
#         self.step_counter = 0

#         # --- FIX: Log a layer that is actually Sparse! ---
#         # Layer 16 is dense, so we log Layer 15 instead.
#         self.logging_layer = 15

#         # Jungle Params
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.current_len = [0] * batch_size

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (Proper SNIS Mode)")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             self.jg_hash_cache = collections.defaultdict(dict)
#         else:
#             print(f"🐷 MagicPIG Attention Enabled")

#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         self.logger.reset()
#         self.step_counter = 0
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         self.step_counter += 1
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def print_summary(self):
#         method = "Jungle" if self.use_jungle else "MagicPIG"
#         self.logger.summary(method)

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         if layer_idx not in self.dense_layers:
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k
#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float()
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         # Init counters
#         log_sink = 0
#         log_local = 0
#         log_sparse = 0
#         log_depths = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 # Dense Layers (We don't log here anymore)
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))
#             else:
#                 # Sparse Layers (This logic runs for Layer 15)
#                 head_outputs = []
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 if self.use_jungle:
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
#                     local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1)
#                             max_d, _ = depths.max(dim=-1)

#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())
#                                 if take > 0:
#                                     sparse_indices = valid_idx[:take] + self.num_sink_tokens
#                                     retrieved_depths = sorted_d[:take].float()
#                                     if h == 0 and req_id == 0:
#                                         log_depths.extend(retrieved_depths.tolist())
#                     else:
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

#                     # Update Logging Counts
#                     if h == 0 and req_id == 0:
#                         log_sink = sink_indices.numel()
#                         log_local = local_indices.numel()
#                         log_sparse = sparse_indices.numel()

#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     depth_map = torch.zeros(full_indices.shape[0], device=self.device)
#                     if self.use_jungle and sparse_indices.numel() > 0:
#                         sparse_idx_to_depth = dict(zip(sparse_indices.tolist(), retrieved_depths.tolist()))
#                         depth_list = [sparse_idx_to_depth.get(idx.item(), 0.0) for idx in full_indices]
#                         depth_map = torch.tensor(depth_list, device=self.device)

#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

#                     if self.use_jungle:
#                         out_h = jungle_snis_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, depth_map, self.jg_L, self.head_dim, is_exact=is_exact)
#                     else:
#                         out_h = magicpig_transform(q_heads[h].unsqueeze(0), k_sel_centered, v_sel, k_norm_sel, self.K, self.L, self.head_dim, is_exact=is_exact)
#                     head_outputs.append(out_h)
#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         # Commit logs (Use self.logging_layer instead of 16)
#         if layer_idx == self.logging_layer and self.step_counter % self.log_interval == 0:
#             self.logger.log("total_tokens", self.current_len[0])
#             self.logger.log("sink_tokens", log_sink)
#             self.logger.log("local_tokens", log_local)
#             self.logger.log("sparse_tokens", log_sparse)
#             if self.use_jungle and log_depths:
#                 self.logger.log("avg_depth", np.mean(log_depths))
#                 self.logger.log("max_depth", np.max(log_depths))

#         return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
# # ==========================================
# # Part 6: Model Wrappers
# # ==========================================

# class LLMLayer:
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device
#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)
#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim
#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             for i in range(bsz):
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down
#         return hidden_states

# class LLM:
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length
#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon
#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)
#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()
#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.clear()
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
#         print("Prefilling...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
#         generated = []
#         curr_pos = seq_len
#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")
#         print("Generating...")
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token
#             position_ids = torch.tensor([[curr_pos]], device=self.device)
#             hidden_states = F.embedding(input_ids, self.embed_tokens)
#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
#             self.attn_server.step()
#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1
#             if next_token.item() in [128001, 128009]:
#                 break

#         # PRINT SUMMARY AT THE END
#         self.attn_server.print_summary()
#         return generated

# if __name__ == "__main__":
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     prompt = "Answer very concisely. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is "
#     tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#     input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#     # 1. RUN MAGICPIG
#     print("\n\n" + "#"*40)
#     print("RUNNING MAGICPIG (LSH)")
#     print("#"*40)
#     llm_mp = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)
#     out_mp = llm_mp.generate(input_ids, max_tokens=20, verbose=False)
#     print(f"Generated (MagicPIG): {tokenizer.decode(out_mp)}")

#     # 2. RUN JUNGLE
#     print("\n\n" + "#"*40)
#     print("RUNNING JUNGLE (Proper SNIS)")
#     print("#"*40)
#     llm_jg = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)
#     out_jg = llm_jg.generate(input_ids, max_tokens=20, verbose=False)
#     print(f"Generated (Jungle): {tokenizer.decode(out_jg)}")

In [7]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()
#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
#     mask = cumulative_probs > top_p
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False
#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)
#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation (Legacy MagicPIG)
# # ==========================================

# class LSH:
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device

#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()
#         results = []
#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups
#             counts = collections.defaultdict(int)
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1
#             candidates = [idx for idx, count in counts.items() if count >= 2]
#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)
#             results.append(t_cand)
#         return results

# # ==========================================
# # Part 3: Sparse Attention Math (SNIS Kernels)
# # ==========================================

# def magicpig_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     """
#     Standard MagicPIG SNIS using fixed K.
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()

#     # Angular Probability
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi

#     # MagicPIG Formula: p^K
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob) # 2-hit approximation
#     log_w = torch.log(w + 1e-4)

#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# def jungle_snis_transform(q, k, v, k_norm, retrieval_depths, L, head_dim, is_exact=None):
#     """

#     Proper SNIS for Jungle Attention.
#     Instead of fixed K, we use the specific `retrieval_depths` per token.
#     w = 1 - (1 - p^d)^L
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Raw Attention Scores
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)

#     # 2. Angular Probability (Base p)
#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)
#     theta = torch.acos(cos_theta)
#     p_base = 1.0 - theta / math.pi

#     # 3. Variable-Depth Collision Probability
#     # The probability of this key colliding with Q at this specific depth d
#     # P(collision) = p_base ^ depth
#     p_collision = p_base.pow(retrieval_depths.float())

#     # 4. Forest Aggregation
#     # Probability of colliding in at least one of L trees
#     # w = 1 - (1 - p_collision)^L
#     w = 1.0 - (1.0 - p_collision).pow(L)
#     log_w = torch.log(w + 1e-4)

#     # 5. SNIS Correction
#     if is_exact is not None:
#         # Exact tokens (Sink/Local) were not found via tree, so bias = 0 (log_w = 0)
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w

#     # 6. Self-Normalization
#     attn_probs = torch.softmax(score_f, dim=0)
#     return torch.matmul(attn_probs.unsqueeze(0), v.float()).to(v.dtype)

# # ==========================================
# # Part 4: Attention Server
# # ==========================================

# class LSHSparseAttnServer:
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose
#         self.use_jungle = use_jungle

#         # Jungle Params
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         # KV Caches
#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.current_len = [0] * batch_size

#         # MagicPIG Component
#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         # Jungle Component
#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (Proper SNIS Mode)")
#             # Projections for Forest of Trees
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             self.jg_hash_cache = collections.defaultdict(dict)

#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len

#         # Update KV Cache
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         # Update Sparse Index
#         if layer_idx not in self.dense_layers:
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k
#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     # Jungle Hashing: Forest of Bit Strings
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float()
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)

#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     # MagicPIG Hashing
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long().permute(0, 2, 1)
#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape

#         # Update Cache
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             # 1. DENSE ATTENTION
#             if layer_idx in self.dense_layers:
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))

#             # 2. SPARSE ATTENTION (Jungle or MagicPIG)
#             else:
#                 head_outputs = []
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 # Pre-calculate Hash Projections
#                 if self.use_jungle:
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float().view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     # Define Regions
#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)
#                     local_indices = torch.arange(max(sparse_boundary, 0), curr_len, device=self.device) if curr_len > sparse_boundary else torch.empty(0, dtype=torch.long, device=self.device)

#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     retrieved_depths = torch.empty(0, dtype=torch.float32, device=self.device)

#                     # --- RETRIEVAL ---
#                     if self.use_jungle:
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]

#                             # Prefix Matching (Depth)
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1) # [N_sparse, L]
#                             max_d, _ = depths.max(dim=-1) # Best depth across forest

#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())

#                                 if take > 0:
#                                     sparse_indices = valid_idx[:take] + self.num_sink_tokens
#                                     # CAPTURE DEPTHS FOR SNIS:
#                                     retrieved_depths = sorted_d[:take].float()
#                     else:
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             sparse_indices = raw_indices[(raw_indices >= self.num_sink_tokens) & (raw_indices < sparse_boundary)]

#                     # --- ESTIMATION ---
#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     # Ensure indices match
#                     # Note: Unique sorts indices, so we must align depths.
#                     # For simplicity in this demo, we will re-map depths or assume sparse region is continuous.
#                     # To be perfectly precise, we build a "Depth Map".

#                     depth_map = torch.zeros(full_indices.shape[0], device=self.device)
#                     if self.use_jungle and sparse_indices.numel() > 0:
#                         # Map retrieved depths to the full index list
#                         # This works because sparse_indices are a subset of full_indices
#                         # We use a scatter approach or dictionary. Here is a tensor mask approach:
#                         mask_sparse = torch.isin(full_indices, sparse_indices)
#                         # Warning: torch.isin doesn't preserve order needed for mapping values.
#                         # Faster approach: Default depth = 0 (bias=0).
#                         # We only need depths for sparse items.
#                         # Since `sparse_indices` came from sort, let's create a lookup.
#                         # (Optimized for readability here, could be faster)

#                         # Re-create sparse indices to depth mapping
#                         sparse_idx_to_depth = dict(zip(sparse_indices.tolist(), retrieved_depths.tolist()))
#                         depth_list = [sparse_idx_to_depth.get(idx.item(), 0.0) for idx in full_indices]
#                         depth_map = torch.tensor(depth_list, device=self.device)

#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]

#                     # Center Keys (Required for Angular Physics in both methods now)
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)

#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)

#                     if self.use_jungle:
#                         # Call Proper Jungle SNIS
#                         out_h = jungle_snis_transform(
#                             q_heads[h].unsqueeze(0),
#                             k_sel_centered,
#                             v_sel,
#                             k_norm_sel,
#                             depth_map, # Pass specific depths
#                             self.jg_L,
#                             self.head_dim,
#                             is_exact=is_exact
#                         )
#                     else:
#                         # Call MagicPIG SNIS
#                         out_h = magicpig_transform(
#                             q_heads[h].unsqueeze(0),
#                             k_sel_centered,
#                             v_sel,
#                             k_norm_sel,
#                             self.K, self.L, self.head_dim,
#                             is_exact=is_exact
#                         )
#                     head_outputs.append(out_h)
#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))
#         return torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)

# # ==========================================
# # Part 5: Model Wrappers (Standard)
# # ==========================================

# class LLMLayer:
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device
#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)
#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)
#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)
#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)
#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim
#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             for i in range(bsz):
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())
#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down
#         return hidden_states

# class LLM:
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length
#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon
#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)
#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()
#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)
#         print("Prefilling...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)
#         generated = []
#         curr_pos = seq_len
#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")
#         print("Generating...")
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token
#             position_ids = torch.tensor([[curr_pos]], device=self.device)
#             hidden_states = F.embedding(input_ids, self.embed_tokens)
#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)
#             self.attn_server.step()
#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1
#             if next_token.item() in [128001, 128009]:
#                 break
#         return generated

# if __name__ == "__main__":
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
#     try:
#         # ENABLE Jungle for SNIS test
#         llm = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)
#         tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#         prompt = "Answer very concisely. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is "
#         print(f"\nPrompt: {prompt}")
#         input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
#         start = time.time()
#         output_ids = llm.generate(input_ids, max_tokens=20, temperature=0.7, verbose=True)
#         end = time.time()
#         decoded = tokenizer.decode(output_ids)
#         print(f"\nGenerated: {decoded}")
#         print(f"Generation Time: {end - start:.2f}s")
#     except Exception as e:
#         print(f"\nError encountered: {e}")

In [8]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     """
#     Equivalent to torch.repeat_interleave(x, dim=1, repeats=n_rep).
#     """
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     """Rotates half the hidden dims of the input."""
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     """
#     Applies Rotary Position Embeddings (RoPE).
#     CRITICAL FIX: Compute in float32 to prevent accumulated noise in deep networks.
#     """
#     # Cast to float32 for rotation
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()

#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     """
#     PyTorch implementation of RMSNorm matching flashinfer/Llama behavior.
#     """
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     """
#     Performs Top-p (nucleus) sampling with temperature.
#     """
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

#     # Remove tokens with cumulative probability above the threshold
#     mask = cumulative_probs > top_p
#     # Shift mask to keep at least one token
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False

#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)

#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation
# # ==========================================

# class LSH:
#     """
#     Python implementation of the LSH logic found in library/lsh/lsh.cc.
#     Implements bucket allocation and the 2-hit filtering retrieval.
#     """
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device

#         # Structure: [layer][req_id][kv_head][l][bucket_hash] -> List[indices]
#         # Using lists instead of fixed tensors allows dynamic sizing (like std::vector in C++)
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         """Clears all hash tables."""
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         """
#         Populates the hash tables.
#         hash_codes: [num_kv_heads, L, seq_len] (int tensors)
#         """
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape

#         # Inefficient in Python but algorithmically correct
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         """
#         Retrieves candidates using the 2-hit filter logic found in lsh.cc.
#         Candidates must appear in at least 2 different hash tables to be considered.
#         query_hash_codes: [batch_size * num_attention_heads, L]
#         """
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()

#         results = []

#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups

#             counts = collections.defaultdict(int)

#             # Count occurrences across L tables
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1

#             # 2-hit filter (matches lsh.cc logic)
#             candidates = [idx for idx, count in counts.items() if count >= 2]

#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)

#             results.append(t_cand)

#         return results

# # ==========================================
# # Part 3: Sparse Attention Math
# # ==========================================

# def sparse_attention_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     """
#     Implements `transform_kernel` from sparse_attention.cc.
#     CRITICAL FIX: Perform dot product and accumulation in float32.
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute scores in Float32 (Fix for precision)
#     # q: [1, D], k: [N, D] -> score: [N]
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)

#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()

#     # 2. MagicPIG Probability Transform (approximate angular distance)
#     # This logic matches sparse_attention.cc/transform_kernel
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     # Clamp for numerical stability in acos
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)

#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)

#     log_w = torch.log(w + 1e-4)

#     # 3. Apply Penalty
#     # Mask penalty for Exact tokens (Sink/Local) - they don't use LSH sampling
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w

#     # 4. Softmax
#     attn_probs = torch.softmax(score_f, dim=0) # [N]

#     # 5. Weighted Sum in Float32 (Fix for precision)
#     # attn_probs: [N], v: [N, D]
#     output = torch.matmul(attn_probs.unsqueeze(0), v.float()) # [1, D]

#     return output.to(v.dtype)

# def jungle_attention_transform(q, k, v, u, head_dim):
#     """
#     Self-Normalized Importance Sampling (SNIS) for Jungle Attention.
#     u: retrieval probability [N]
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute Raw Scores (Dot Product)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0) / math.sqrt(head_dim)

#     # 2. Stable Exp
#     score_max = score_f.max()
#     score_f = score_f - score_max
#     w = torch.exp(score_f)

#     # 3. Importance Sampling Correction
#     # w_hat = w / u
#     w_hat = w / (u + 1e-6)

#     # 4. Normalized Weighted Sum
#     # Output = sum(w_hat * v) / sum(w_hat)
#     denominator = w_hat.sum() + 1e-9
#     attn_probs = w_hat / denominator

#     output = torch.matmul(attn_probs.unsqueeze(0), v.float())
#     return output.to(v.dtype)

# # ==========================================
# # Part 4: Attention Server
# # ==========================================

# class LSHSparseAttnServer:
#     """
#     Manages KV caching, LSH tables, and the hybrid attention mechanism.
#     Coordinates between Dense (Sink/Local) and Sparse (LSH) attention.
#     """
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  # Jungle Params
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose

#         # Jungle config
#         self.use_jungle = use_jungle
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         # KV Caches
#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.current_len = [0] * batch_size

#         # Average K (centroid) for the sparse region, used for centering keys before hashing
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)

#         # Random projection matrix for LSH (Standard)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         # Jungle Projections
#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (L={jg_L}, K_max={jg_K_max}, Budget={jg_budget})")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             # Store hash bits for sparse regions [layer][req_id][kv_head] -> Tensor[N_sparse, L, K_max]
#             self.jg_hash_cache = collections.defaultdict(dict)

#         # Track the boundary of the sparse region per request
#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         """
#         Advances the sequence length counter for all requests.
#         Must be called ONCE after all layers have processed the current token.
#         """
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         """
#         Updates KV cache and fills LSH tables for the 'offload' region (Sparse).
#         Corresponds to attnserver.py fill logic.
#         """
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len

#         # 1. Update KV Cache (Store RAW keys)
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)

#         # Update current length during prefill
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         # 2. Update LSH Tables (only for sparse layers during Pre-fill)
#         if layer_idx not in self.dense_layers:
#             # Logic matches attnserver.py: index tokens between [sink, end-local]
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens

#             # Store boundary: tokens before this are Sparse (hashed), after are Local (Exact)
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             # Only index if there is a sparse region
#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]

#                 # Center keys using mean of the sparse region (float32 for precision)
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k

#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     # --- Jungle Hashing ---
#                     # Compute L*K_max bits for Jungle Forest
#                     # [num_kv, N, D] @ [D, L*K_max] -> [num_kv, N, L*K_max]
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float() # [num_kv, N, L*K_max]

#                     # Reshape to [num_kv, N, L, K_max]
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)

#                     # Store in cache
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     # --- Standard LSH ---
#                     # Compute LSH hashes
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long()
#                     buckets = buckets.permute(0, 2, 1)

#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)
#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         assert q_len == 1

#         # --- Update Cache ---
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             # 1. DENSE LAYERS (Standard Attention)
#             if layer_idx in self.dense_layers:
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)

#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))

#             # 2. SPARSE LAYERS (LSH or Jungle)
#             else:
#                 head_outputs = []

#                 # --- Shared Pre-computation ---
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 # Pre-calculate retrieval structures
#                 if self.use_jungle:
#                     # Jungle: Project Q to L*K_max bits
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     # LSH: Project Q to L*K bits and bucket
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     # LSH Batch Retrieve happens here
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     # --- A. Define Exact Regions (Sink & Local) ---
#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)

#                     if curr_len > sparse_boundary:
#                         local_start = max(sparse_boundary, 0)
#                         local_indices = torch.arange(local_start, curr_len, device=self.device)
#                     else:
#                         local_indices = torch.empty(0, dtype=torch.long, device=self.device)

#                     # --- B. Retrieval Strategy (The ONLY Difference) ---
#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)

#                     if self.use_jungle:
#                         # === STRATEGY: JUNGLE TREE ===
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             # Retrieve cached K bits
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]

#                             # Prefix Match (Depth Calculation)
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1) # [N_sparse, L]
#                             max_d, _ = depths.max(dim=-1)

#                             # Budget Selection
#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)

#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())

#                                 if take > 0:
#                                     rel_indices = valid_idx[:take]
#                                     sparse_indices = rel_indices + self.num_sink_tokens
#                     else:
#                         # === STRATEGY: MAGICPIG LSH ===
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             # Filter to strictly sparse region
#                             sparse_indices = raw_indices[
#                                 (raw_indices >= self.num_sink_tokens) &
#                                 (raw_indices < sparse_boundary)
#                             ]

#                     # --- C. Unified Estimation (MagicPIG Kernel) ---
#                     # Combine all indices
#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     # Fetch Keys/Values
#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]

#                     # Center Keys (Required for Angular Kernel)
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k

#                     # Mask for Exact Tokens (Sink/Local get normal softmax, Sparse gets kernel correction)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)

#                     # Use MagicPIG Kernel for BOTH
#                     # Note: We pass self.K/self.L even for Jungle to mimic behavior
#                     out_h = sparse_attention_transform(
#                         q_heads[h].unsqueeze(0),
#                         k_sel_centered,
#                         v_sel,
#                         k_norm_sel,
#                         self.K, self.L, self.head_dim,
#                         is_exact=is_exact
#                     )

#                     head_outputs.append(out_h)

#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         final_out = torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
#         return final_out

# # ==========================================
# # Part 5: Model Wrappers
# # ==========================================

# class LLMLayer:
#     """
#     Wraps a HuggingFace LlamaDecoderLayer.
#     Matches the structure of models/llama.py.
#     """
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device

#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)

#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)

#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)

#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)

#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim

#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)

#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             # Prefill: Use Dense Attention & Build LSH Index
#             for i in range(bsz):
#                 # Call fill to populate LSH with the Prompt's sparse region
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())

#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             # Use PyTorch SDPA for dense prefill (robust implementation)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             # Decode: Use Sparse Attention
#             # Passes current token k, v to decode for local caching
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states

#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down

#         return hidden_states

# class LLM:
#     """
#     Main Model Class.
#     Matches models/llama.py structure.
#     """
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length

#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon

#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)

#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()

#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)

#         print("Prefilling (Dense Attention)...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)

#         # Prefill Phase
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)

#         generated = []
#         curr_pos = seq_len

#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")

#         print("Generating (Sparse Attention)...")
#         # Decode Phase
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token # [bsz, 1]
#             position_ids = torch.tensor([[curr_pos]], device=self.device)

#             hidden_states = F.embedding(input_ids, self.embed_tokens)

#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)

#             # --- CRITICAL FIX: Update Sequence Length ONCE per step ---
#             self.attn_server.step()

#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1

#             # Stop tokens for Llama 3
#             if next_token.item() in [128001, 128009]:
#                 break

#         return generated

# # ==========================================
# # Main Execution
# # ==========================================

# if __name__ == "__main__":
#     # Example Usage
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     try:
#         # Initialize model with Jungle Attention Enabled
#         # Set use_jungle=True to test the new logic
#         llm = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)

#         tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#         prompt = "Answer very concisely. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is "


#         print(f"\nPrompt: {prompt}")
#         input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#         # Generate with Verbose Debugging
#         start = time.time()
#         output_ids = llm.generate(input_ids, max_tokens=10, temperature=0.7, verbose=True)
#         end = time.time()

#         decoded = tokenizer.decode(output_ids)
#         print(f"\nGenerated: {decoded}")
#         print(f"Generation Time: {end - start:.2f}s")

#     except Exception as e:
#         print(f"\nError encountered: {e}")

In [9]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     """
#     Equivalent to torch.repeat_interleave(x, dim=1, repeats=n_rep).
#     """
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     """Rotates half the hidden dims of the input."""
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     """
#     Applies Rotary Position Embeddings (RoPE).
#     CRITICAL FIX: Compute in float32 to prevent accumulated noise in deep networks.
#     """
#     # Cast to float32 for rotation
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()

#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     """
#     PyTorch implementation of RMSNorm matching flashinfer/Llama behavior.
#     """
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     """
#     Performs Top-p (nucleus) sampling with temperature.
#     """
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

#     # Remove tokens with cumulative probability above the threshold
#     mask = cumulative_probs > top_p
#     # Shift mask to keep at least one token
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False

#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)

#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation
# # ==========================================

# class LSH:
#     """
#     Python implementation of the LSH logic found in library/lsh/lsh.cc.
#     Implements bucket allocation and the 2-hit filtering retrieval.
#     """
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device

#         # Structure: [layer][req_id][kv_head][l][bucket_hash] -> List[indices]
#         # Using lists instead of fixed tensors allows dynamic sizing (like std::vector in C++)
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         """Clears all hash tables."""
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         """
#         Populates the hash tables.
#         hash_codes: [num_kv_heads, L, seq_len] (int tensors)
#         """
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape

#         # Inefficient in Python but algorithmically correct
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         """
#         Retrieves candidates using the 2-hit filter logic found in lsh.cc.
#         Candidates must appear in at least 2 different hash tables to be considered.
#         query_hash_codes: [batch_size * num_attention_heads, L]
#         """
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()

#         results = []

#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups

#             counts = collections.defaultdict(int)

#             # Count occurrences across L tables
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1

#             # 2-hit filter (matches lsh.cc logic)
#             candidates = [idx for idx, count in counts.items() if count >= 2]

#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)

#             results.append(t_cand)

#         return results

# # ==========================================
# # Part 3: Sparse Attention Math
# # ==========================================

# def sparse_attention_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     """
#     Implements `transform_kernel` from sparse_attention.cc.
#     CRITICAL FIX: Perform dot product and accumulation in float32.
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute scores in Float32 (Fix for precision)
#     # q: [1, D], k: [N, D] -> score: [N]
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)

#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()

#     # 2. MagicPIG Probability Transform (approximate angular distance)
#     # This logic matches sparse_attention.cc/transform_kernel
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     # Clamp for numerical stability in acos
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)

#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)

#     log_w = torch.log(w + 1e-4)

#     # 3. Apply Penalty
#     # Mask penalty for Exact tokens (Sink/Local) - they don't use LSH sampling
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w

#     # 4. Softmax
#     attn_probs = torch.softmax(score_f, dim=0) # [N]

#     # 5. Weighted Sum in Float32 (Fix for precision)
#     # attn_probs: [N], v: [N, D]
#     output = torch.matmul(attn_probs.unsqueeze(0), v.float()) # [1, D]

#     return output.to(v.dtype)

# def jungle_attention_transform(q, k, v, u, head_dim):
#     """
#     Self-Normalized Importance Sampling (SNIS) for Jungle Attention.
#     u: retrieval probability [N]
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute Raw Scores (Dot Product)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0) / math.sqrt(head_dim)

#     # 2. Stable Exp
#     score_max = score_f.max()
#     score_f = score_f - score_max
#     w = torch.exp(score_f)

#     # 3. Importance Sampling Correction
#     # w_hat = w / u
#     w_hat = w / (u + 1e-6)

#     # 4. Normalized Weighted Sum
#     # Output = sum(w_hat * v) / sum(w_hat)
#     denominator = w_hat.sum() + 1e-9
#     attn_probs = w_hat / denominator

#     output = torch.matmul(attn_probs.unsqueeze(0), v.float())
#     return output.to(v.dtype)

# # ==========================================
# # Part 4: Attention Server
# # ==========================================

# class LSHSparseAttnServer:
#     """
#     Manages KV caching, LSH tables, and the hybrid attention mechanism.
#     Coordinates between Dense (Sink/Local) and Sparse (LSH) attention.
#     """
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  # Jungle Params
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose

#         # Jungle config
#         self.use_jungle = use_jungle
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         # KV Caches
#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.current_len = [0] * batch_size

#         # Average K (centroid) for the sparse region, used for centering keys before hashing
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)

#         # Random projection matrix for LSH (Standard)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         # Jungle Projections
#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (L={jg_L}, K_max={jg_K_max}, Budget={jg_budget})")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             # Store hash bits for sparse regions [layer][req_id][kv_head] -> Tensor[N_sparse, L, K_max]
#             self.jg_hash_cache = collections.defaultdict(dict)

#         # Track the boundary of the sparse region per request
#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         """
#         Advances the sequence length counter for all requests.
#         Must be called ONCE after all layers have processed the current token.
#         """
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         """
#         Updates KV cache and fills LSH tables for the 'offload' region (Sparse).
#         Corresponds to attnserver.py fill logic.
#         """
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len

#         # 1. Update KV Cache (Store RAW keys)
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)

#         # Update current length during prefill
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         # 2. Update LSH Tables (only for sparse layers during Pre-fill)
#         if layer_idx not in self.dense_layers:
#             # Logic matches attnserver.py: index tokens between [sink, end-local]
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens

#             # Store boundary: tokens before this are Sparse (hashed), after are Local (Exact)
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             # Only index if there is a sparse region
#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]

#                 # Center keys using mean of the sparse region (float32 for precision)
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k

#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     # --- Jungle Hashing ---
#                     # Compute L*K_max bits for Jungle Forest
#                     # [num_kv, N, D] @ [D, L*K_max] -> [num_kv, N, L*K_max]
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float() # [num_kv, N, L*K_max]

#                     # Reshape to [num_kv, N, L, K_max]
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)

#                     # Store in cache
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     # --- Standard LSH ---
#                     # Compute LSH hashes
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long()
#                     buckets = buckets.permute(0, 2, 1)

#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)
#     def decode(self, query_states, key_states, value_states, layer_idx):
#         bsz, n_heads, q_len, dim = query_states.shape
#         assert q_len == 1

#         # --- Update Cache ---
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = key_states[req_id]
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = value_states[req_id]

#         hidden_states_list = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             # 1. DENSE LAYERS (Standard Attention)
#             if layer_idx in self.dense_layers:
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)

#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))

#             # 2. SPARSE LAYERS (LSH or Jungle)
#             else:
#                 head_outputs = []

#                 # --- Shared Pre-computation ---
#                 norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)

#                 # Pre-calculate retrieval structures
#                 if self.use_jungle:
#                     # Jungle: Project Q to L*K_max bits
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     # LSH: Project Q to L*K bits and bucket
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     # LSH Batch Retrieve happens here
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     # --- A. Define Exact Regions (Sink & Local) ---
#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)

#                     if curr_len > sparse_boundary:
#                         local_start = max(sparse_boundary, 0)
#                         local_indices = torch.arange(local_start, curr_len, device=self.device)
#                     else:
#                         local_indices = torch.empty(0, dtype=torch.long, device=self.device)

#                     # --- B. Retrieval Strategy (The ONLY Difference) ---
#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)

#                     if self.use_jungle:
#                         # === STRATEGY: JUNGLE TREE ===
#                         if (layer_idx in self.jg_hash_cache and req_id in self.jg_hash_cache[layer_idx]):
#                             # Retrieve cached K bits
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h]

#                             # Prefix Match (Depth Calculation)
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             depths = match.cumprod(dim=-1).sum(dim=-1) # [N_sparse, L]
#                             max_d, _ = depths.max(dim=-1)

#                             # Budget Selection
#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)

#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())

#                                 if take > 0:
#                                     rel_indices = valid_idx[:take]
#                                     sparse_indices = rel_indices + self.num_sink_tokens
#                     else:
#                         # === STRATEGY: MAGICPIG LSH ===
#                         raw_indices = idx_list[h]
#                         if raw_indices.numel() > 0:
#                             # Filter to strictly sparse region
#                             sparse_indices = raw_indices[
#                                 (raw_indices >= self.num_sink_tokens) &
#                                 (raw_indices < sparse_boundary)
#                             ]

#                     # --- C. Unified Estimation (MagicPIG Kernel) ---
#                     # Combine all indices
#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     # Fetch Keys/Values
#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]

#                     # Center Keys (Required for Angular Kernel)
#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k

#                     # Mask for Exact Tokens (Sink/Local get normal softmax, Sparse gets kernel correction)
#                     is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)
#                     k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)

#                     # Use MagicPIG Kernel for BOTH
#                     # Note: We pass self.K/self.L even for Jungle to mimic behavior
#                     out_h = sparse_attention_transform(
#                         q_heads[h].unsqueeze(0),
#                         k_sel_centered,
#                         v_sel,
#                         k_norm_sel,
#                         self.K, self.L, self.head_dim,
#                         is_exact=is_exact
#                     )

#                     head_outputs.append(out_h)

#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         final_out = torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
#         return final_out

# # ==========================================
# # Part 5: Model Wrappers
# # ==========================================

# class LLMLayer:
#     """
#     Wraps a HuggingFace LlamaDecoderLayer.
#     Matches the structure of models/llama.py.
#     """
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device

#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)

#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)

#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)

#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)

#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim

#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)

#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             # Prefill: Use Dense Attention & Build LSH Index
#             for i in range(bsz):
#                 # Call fill to populate LSH with the Prompt's sparse region
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())

#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             # Use PyTorch SDPA for dense prefill (robust implementation)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             # Decode: Use Sparse Attention
#             # Passes current token k, v to decode for local caching
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states

#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down

#         return hidden_states

# class LLM:
#     """
#     Main Model Class.
#     Matches models/llama.py structure.
#     """
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length

#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon

#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)

#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()

#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)

#         print("Prefilling (Dense Attention)...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)

#         # Prefill Phase
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)

#         generated = []
#         curr_pos = seq_len

#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")

#         print("Generating (Sparse Attention)...")
#         # Decode Phase
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token # [bsz, 1]
#             position_ids = torch.tensor([[curr_pos]], device=self.device)

#             hidden_states = F.embedding(input_ids, self.embed_tokens)

#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)

#             # --- CRITICAL FIX: Update Sequence Length ONCE per step ---
#             self.attn_server.step()

#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1

#             # Stop tokens for Llama 3
#             if next_token.item() in [128001, 128009]:
#                 break

#         return generated

# # ==========================================
# # Main Execution
# # ==========================================

# if __name__ == "__main__":
#     # Example Usage
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     try:
#         # Initialize model with Jungle Attention Enabled
#         # Set use_jungle=True to test the new logic
#         llm = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=True)

#         tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#         prompt = "Answer very concisely. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is "


#         print(f"\nPrompt: {prompt}")
#         input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#         # Generate with Verbose Debugging
#         start = time.time()
#         output_ids = llm.generate(input_ids, max_tokens=10, temperature=0.7, verbose=True)
#         end = time.time()

#         decoded = tokenizer.decode(output_ids)
#         print(f"\nGenerated: {decoded}")
#         print(f"Generation Time: {end - start:.2f}s")

#     except Exception as e:
#         print(f"\nError encountered: {e}")

In [10]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     """
#     Equivalent to torch.repeat_interleave(x, dim=1, repeats=n_rep).
#     """
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     """Rotates half the hidden dims of the input."""
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     """
#     Applies Rotary Position Embeddings (RoPE).
#     CRITICAL FIX: Compute in float32 to prevent accumulated noise in deep networks.
#     """
#     # Cast to float32 for rotation
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()

#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     """
#     PyTorch implementation of RMSNorm matching flashinfer/Llama behavior.
#     """
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     """
#     Performs Top-p (nucleus) sampling with temperature.
#     """
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

#     # Remove tokens with cumulative probability above the threshold
#     mask = cumulative_probs > top_p
#     # Shift mask to keep at least one token
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False

#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)

#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation
# # ==========================================

# class LSH:
#     """
#     Python implementation of the LSH logic found in library/lsh/lsh.cc.
#     Implements bucket allocation and the 2-hit filtering retrieval.
#     """
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device

#         # Structure: [layer][req_id][kv_head][l][bucket_hash] -> List[indices]
#         # Using lists instead of fixed tensors allows dynamic sizing (like std::vector in C++)
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         """Clears all hash tables."""
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         """
#         Populates the hash tables.
#         hash_codes: [num_kv_heads, L, seq_len] (int tensors)
#         """
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape

#         # Inefficient in Python but algorithmically correct
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         """
#         Retrieves candidates using the 2-hit filter logic found in lsh.cc.
#         Candidates must appear in at least 2 different hash tables to be considered.
#         query_hash_codes: [batch_size * num_attention_heads, L]
#         """
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()

#         results = []

#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups

#             counts = collections.defaultdict(int)

#             # Count occurrences across L tables
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1

#             # 2-hit filter (matches lsh.cc logic)
#             candidates = [idx for idx, count in counts.items() if count >= 2]

#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)

#             results.append(t_cand)

#         return results

# # ==========================================
# # Part 3: Sparse Attention Math
# # ==========================================

# def sparse_attention_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     """
#     Implements `transform_kernel` from sparse_attention.cc.
#     CRITICAL FIX: Perform dot product and accumulation in float32.
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute scores in Float32 (Fix for precision)
#     # q: [1, D], k: [N, D] -> score: [N]
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)

#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()

#     # 2. MagicPIG Probability Transform (approximate angular distance)
#     # This logic matches sparse_attention.cc/transform_kernel
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     # Clamp for numerical stability in acos
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)

#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)

#     log_w = torch.log(w + 1e-4)

#     # 3. Apply Penalty
#     # Mask penalty for Exact tokens (Sink/Local) - they don't use LSH sampling
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w

#     # 4. Softmax
#     attn_probs = torch.softmax(score_f, dim=0) # [N]

#     # 5. Weighted Sum in Float32 (Fix for precision)
#     # attn_probs: [N], v: [N, D]
#     output = torch.matmul(attn_probs.unsqueeze(0), v.float()) # [1, D]

#     return output.to(v.dtype)

# def jungle_attention_transform(q, k, v, u, head_dim):
#     """
#     Self-Normalized Importance Sampling (SNIS) for Jungle Attention.
#     u: retrieval probability [N]
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute Raw Scores (Dot Product)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0) / math.sqrt(head_dim)

#     # 2. Stable Exp
#     score_max = score_f.max()
#     score_f = score_f - score_max
#     w = torch.exp(score_f)

#     # 3. Importance Sampling Correction
#     # w_hat = w / u
#     w_hat = w / (u + 1e-6)

#     # 4. Normalized Weighted Sum
#     # Output = sum(w_hat * v) / sum(w_hat)
#     denominator = w_hat.sum() + 1e-9
#     attn_probs = w_hat / denominator

#     output = torch.matmul(attn_probs.unsqueeze(0), v.float())
#     return output.to(v.dtype)

# # ==========================================
# # Part 4: Attention Server
# # ==========================================

# class LSHSparseAttnServer:
#     """
#     Manages KV caching, LSH tables, and the hybrid attention mechanism.
#     Coordinates between Dense (Sink/Local) and Sparse (LSH) attention.
#     """
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  # Jungle Params
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose

#         # Jungle config
#         self.use_jungle = use_jungle
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         # KV Caches
#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.current_len = [0] * batch_size

#         # Average K (centroid) for the sparse region, used for centering keys before hashing
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)

#         # Random projection matrix for LSH (Standard)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         # Jungle Projections
#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (L={jg_L}, K_max={jg_K_max}, Budget={jg_budget})")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             # Store hash bits for sparse regions [layer][req_id][kv_head] -> Tensor[N_sparse, L, K_max]
#             self.jg_hash_cache = collections.defaultdict(dict)

#         # Track the boundary of the sparse region per request
#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         """
#         Advances the sequence length counter for all requests.
#         Must be called ONCE after all layers have processed the current token.
#         """
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         """
#         Updates KV cache and fills LSH tables for the 'offload' region (Sparse).
#         Corresponds to attnserver.py fill logic.
#         """
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len

#         # 1. Update KV Cache (Store RAW keys)
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)

#         # Update current length during prefill
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         # 2. Update LSH Tables (only for sparse layers during Pre-fill)
#         if layer_idx not in self.dense_layers:
#             # Logic matches attnserver.py: index tokens between [sink, end-local]
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens

#             # Store boundary: tokens before this are Sparse (hashed), after are Local (Exact)
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             # Only index if there is a sparse region
#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]

#                 # Center keys using mean of the sparse region (float32 for precision)
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k

#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     # --- Jungle Hashing ---
#                     # Compute L*K_max bits for Jungle Forest
#                     # [num_kv, N, D] @ [D, L*K_max] -> [num_kv, N, L*K_max]
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float() # [num_kv, N, L*K_max]

#                     # Reshape to [num_kv, N, L, K_max]
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)

#                     # Store in cache
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     # --- Standard LSH ---
#                     # Compute LSH hashes
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long()
#                     buckets = buckets.permute(0, 2, 1)

#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         """
#         Performs decoding step with DEBUG STATEMENTS added.
#         """
#         bsz, n_heads, q_len, dim = query_states.shape
#         assert q_len == 1, "Sparse decode expects query length 1"

#         # --- Update Cache with new token ---
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             k_slice = key_states[req_id]
#             v_slice = value_states[req_id]

#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = k_slice
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = v_slice

#         hidden_states_list = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 # ... (Dense logic remains the same) ...
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)

#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))

#             else:
#                 # --- Sparse Retrieval Logic ---
#                 head_outputs = []

#                 # Pre-compute Q hash for Jungle if enabled
#                 if self.use_jungle:
#                     norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)
#                     # [Heads, D] @ [D, L*K] -> [Heads, L*K]
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     # 1. Sink Tokens
#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)

#                     # 2. Local Tokens
#                     if curr_len > sparse_boundary:
#                         local_start = max(sparse_boundary, 0)
#                         local_indices = torch.arange(local_start, curr_len, device=self.device)
#                     else:
#                         local_indices = torch.empty(0, dtype=torch.long, device=self.device)

#                     # 3. Sparse Tokens
#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     sparse_u = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         # --- Jungle Retrieval ---
#                         # Check if we have sparse keys for this request/layer
#                         if (layer_idx in self.jg_hash_cache and
#                             req_id in self.jg_hash_cache[layer_idx]):

#                             # Get cached K bits: [N_sparse, L, K_max]
#                             # Note: self.jg_hash_cache stores [num_kv_heads, N, L, K]
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h] # [L, K_max]

#                             # Compute prefix matches (depths)
#                             # (q == k) -> [N, L, K]
#                             # cumprod to find prefix length
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             # cumprod over K dim, sum over K dim -> depth
#                             depths = match.cumprod(dim=-1).sum(dim=-1) # [N, L]

#                             # Max depth across L trees
#                             max_d, _ = depths.max(dim=-1) # [N]

#                             # Select Top-B based on max_d
#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 # Apply min depth filter
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())

#                                 if take > 0:
#                                     # sparse_indices_raw are relative to sparse region start
#                                     # Sparse region starts at num_sink_tokens
#                                     rel_indices = valid_idx[:take]
#                                     sparse_indices = rel_indices + self.num_sink_tokens

#                                     # Compute U weights
#                                     # d_thresh is the depth of the worst selected item (approximation)
#                                     d_thresh = max_d[rel_indices].min().float()

#                                     # Need Cosine Sim for p
#                                     # Re-compute dot product for selected keys?
#                                     # Approximating p from depth is hard without calibration.
#                                     # Jungle paper suggests estimating p based on d.
#                                     # Or we compute exact dot product for selected keys to get p.
#                                     # Let's compute exact p for the selected set to be precise for SNIS.
#                                     k_sel_sparse = self.k_cache[layer_idx][req_id, kv_head, sparse_indices, :]
#                                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]

#                                     # Normalize centered
#                                     Q_norm_h = norm_q[h]
#                                     K_centered_norm = F.normalize(k_sel_sparse - avg_k, dim=-1)
#                                     dot = (Q_norm_h @ K_centered_norm.T).clamp(-0.999, 0.999)
#                                     p_est = 1.0 - (torch.acos(dot) / math.pi)

#                                     p_d = p_est.pow(d_thresh)
#                                     sparse_u = (1.0 - (1.0 - p_d).pow(self.jg_L)).clamp(1e-6, 1.0)

#                     else:
#                         # --- Standard LSH Retrieval ---
#                         sparse_indices_raw = idx_list[h]
#                         if sparse_indices_raw.numel() > 0:
#                             sparse_indices = sparse_indices_raw[(sparse_indices_raw >= self.num_sink_tokens) & (sparse_indices_raw < sparse_boundary)]
#                             # Standard LSH weights imply u depends on collision prob, usually calculated inside transform or assumed
#                             # Here we just pass ones for sparse_u later if needed, but standard LSH code
#                             # usually applies a different correction. The helper `sparse_attention_transform`
#                             # computes the magicpig probability internally.

#                     # Combine indices
#                     # full_indices: [Sink..., Sparse..., Local...]
#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     # ========================================================
#                     # DEBUG START: Sparsity Visualization
#                     # ========================================================
#                     if self.verbose and h == 0 and req_id == 0:
#                         total_tokens = curr_len
#                         kept = full_indices.numel()

#                         if total_tokens > (self.num_sink_tokens + self.num_local_tokens):
#                             ignored = total_tokens - kept
#                             sparsity_ratio = (ignored / total_tokens) * 100

#                             method_name = "Jungle" if self.use_jungle else "MagicPIG"
#                             print(f"\n[DEBUG] {method_name} | Layer {layer_idx} | SeqLen {total_tokens}")
#                             print(f"  Status:  Attending to {kept} tokens (Ignored {ignored})")
#                             print(f"  Sparsity: {sparsity_ratio:.2f}%")

#                             if self.use_jungle and sparse_indices.numel() > 0:
#                                 print(f"  Jungle:  Selected {sparse_indices.numel()} sparse tokens")
#                                 print(f"           Min Weight u: {sparse_u.min().item():.4f}")
#                     # ========================================================

#                     # Prepare for attention transform
#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]

#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k

#                     q_h = q_heads[h].unsqueeze(0)

#                     if self.use_jungle:
#                         # Construct full U vector
#                         # Sink/Local have u=1.0
#                         # Sparse have calculated u
#                         # We need to map sparse_u back to full_indices positions
#                         # This is tricky with unique(). Simplification:
#                         # Re-construct indices without unique to align u
#                         # (Assume disjoint sets for simplicity or just assign 1 to non-sparse)

#                         # Create a map for u values
#                         u_map = torch.ones(curr_len, device=self.device, dtype=torch.float32)
#                         if sparse_indices.numel() > 0:
#                             # CRITICAL FIX: Cast sparse_u to float32 to match u_map dtype
#                             u_map[sparse_indices] = sparse_u.to(torch.float32)

#                         u_vec = u_map[full_indices]

#                         out_h = jungle_attention_transform(
#                             q_h, k_sel_centered, v_sel, u_vec, self.head_dim
#                         )
#                     else:
#                         is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)
#                         k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                         out_h = sparse_attention_transform(
#                             q_h, k_sel_centered, v_sel, k_norm_sel,
#                             self.K, self.L, self.head_dim,
#                             is_exact=is_exact
#                         )

#                     head_outputs.append(out_h)

#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         final_out = torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
#         return final_out

# # ==========================================
# # Part 5: Model Wrappers
# # ==========================================

# class LLMLayer:
#     """
#     Wraps a HuggingFace LlamaDecoderLayer.
#     Matches the structure of models/llama.py.
#     """
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device

#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)

#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)

#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)

#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)

#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim

#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)

#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             # Prefill: Use Dense Attention & Build LSH Index
#             for i in range(bsz):
#                 # Call fill to populate LSH with the Prompt's sparse region
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())

#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             # Use PyTorch SDPA for dense prefill (robust implementation)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             # Decode: Use Sparse Attention
#             # Passes current token k, v to decode for local caching
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states

#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down

#         return hidden_states

# class LLM:
#     """
#     Main Model Class.
#     Matches models/llama.py structure.
#     """
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length

#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon

#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)

#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()

#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)

#         print("Prefilling (Dense Attention)...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)

#         # Prefill Phase
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)

#         generated = []
#         curr_pos = seq_len

#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")

#         print("Generating (Sparse Attention)...")
#         # Decode Phase
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token # [bsz, 1]
#             position_ids = torch.tensor([[curr_pos]], device=self.device)

#             hidden_states = F.embedding(input_ids, self.embed_tokens)

#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)

#             # --- CRITICAL FIX: Update Sequence Length ONCE per step ---
#             self.attn_server.step()

#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1

#             # Stop tokens for Llama 3
#             if next_token.item() in [128001, 128009]:
#                 break

#         return generated

# # ==========================================
# # Main Execution
# # ==========================================

# if __name__ == "__main__":
#     # Example Usage
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     try:
#         # Initialize model with Jungle Attention Enabled
#         # Set use_jungle=True to test the new logic
#         llm = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)

#         tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#         prompt = "Answer very concisely. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total? The total is "


#         print(f"\nPrompt: {prompt}")
#         input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#         # Generate with Verbose Debugging
#         start = time.time()
#         output_ids = llm.generate(input_ids, max_tokens=10, temperature=0.7, verbose=False)
#         end = time.time()

#         decoded = tokenizer.decode(output_ids)
#         print(f"\nGenerated: {decoded}")
#         print(f"Generation Time: {end - start:.2f}s")

#     except Exception as e:
#         print(f"\nError encountered: {e}")

In [11]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from transformers import LlamaForCausalLM, LlamaConfig, AutoTokenizer
# import math
# import collections
# import time
# import gc

# # ==========================================
# # Part 1: Utility Functions
# # ==========================================

# def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
#     """
#     Equivalent to torch.repeat_interleave(x, dim=1, repeats=n_rep).
#     """
#     batch, num_key_value_heads, slen, head_dim = hidden_states.shape
#     if n_rep == 1:
#         return hidden_states
#     hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
#     return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

# def rotate_half(x):
#     """Rotates half the hidden dims of the input."""
#     x1 = x[..., : x.shape[-1] // 2]
#     x2 = x[..., x.shape[-1] // 2 :]
#     return torch.cat((-x2, x1), dim=-1)

# def apply_rotary_pos_emb(q, cos, sin, position_ids, unsqueeze_dim=1):
#     """
#     Applies Rotary Position Embeddings (RoPE).
#     CRITICAL FIX: Compute in float32 to prevent accumulated noise in deep networks.
#     """
#     # Cast to float32 for rotation
#     q_f32 = q.float()
#     cos = cos[position_ids].unsqueeze(unsqueeze_dim).float()
#     sin = sin[position_ids].unsqueeze(unsqueeze_dim).float()

#     q_embed = (q_f32 * cos) + (rotate_half(q_f32) * sin)
#     return q_embed.to(q.dtype)

# def manual_rmsnorm(hidden_states, weight, variance_epsilon):
#     """
#     PyTorch implementation of RMSNorm matching flashinfer/Llama behavior.
#     """
#     input_dtype = hidden_states.dtype
#     hidden_states = hidden_states.to(torch.float32)
#     variance = hidden_states.pow(2).mean(-1, keepdim=True)
#     hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
#     return weight * hidden_states.to(input_dtype)

# def topp_temperature_decode(logits, temperature=0.6, top_p=0.9):
#     """
#     Performs Top-p (nucleus) sampling with temperature.
#     """
#     logits = logits / temperature
#     probs = torch.softmax(logits, dim=-1)
#     sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
#     cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

#     # Remove tokens with cumulative probability above the threshold
#     mask = cumulative_probs > top_p
#     # Shift mask to keep at least one token
#     mask[:, :, 1:] = mask[:, :, :-1].clone()
#     mask[:, :, 0] = False

#     sorted_probs.masked_fill_(mask, 0.0)
#     sorted_probs /= sorted_probs.sum(dim=-1, keepdim=True)

#     sampled_indices = torch.multinomial(sorted_probs.squeeze(1), num_samples=1)
#     final_indices = sorted_indices.gather(dim=-1, index=sampled_indices.unsqueeze(-1))
#     return final_indices.squeeze(-1)

# # ==========================================
# # Part 2: LSH Implementation
# # ==========================================

# class LSH:
#     """
#     Python implementation of the LSH logic found in library/lsh/lsh.cc.
#     Implements bucket allocation and the 2-hit filtering retrieval.
#     """
#     def __init__(self, K, L, num_layers, num_heads, num_kv_heads, batch_size, max_length, device='cuda:0'):
#         self.K = K
#         self.L = L
#         self.num_layers = num_layers
#         self.num_heads = num_heads
#         self.num_kv_heads = num_kv_heads
#         self.batch_size = batch_size
#         self.max_length = max_length
#         self.num_attention_groups = num_heads // num_kv_heads
#         self.device = device

#         # Structure: [layer][req_id][kv_head][l][bucket_hash] -> List[indices]
#         # Using lists instead of fixed tensors allows dynamic sizing (like std::vector in C++)
#         self.tables = []
#         for _ in range(num_layers):
#             req_tables = []
#             for _ in range(batch_size):
#                 head_tables = []
#                 for _ in range(num_kv_heads):
#                     l_tables = [collections.defaultdict(list) for _ in range(L)]
#                     head_tables.append(l_tables)
#                 req_tables.append(head_tables)
#             self.tables.append(req_tables)

#     def clear(self):
#         """Clears all hash tables."""
#         for layer_idx in range(self.num_layers):
#             for req_id in range(self.batch_size):
#                 for head_idx in range(self.num_kv_heads):
#                     for l in range(self.L):
#                         self.tables[layer_idx][req_id][head_idx][l].clear()

#     def fill(self, layer_id, request_id, hash_codes, indices):
#         """
#         Populates the hash tables.
#         hash_codes: [num_kv_heads, L, seq_len] (int tensors)
#         """
#         hc = hash_codes.cpu()
#         idx = indices.cpu()
#         num_kv, L, seq_len = hc.shape

#         # Inefficient in Python but algorithmically correct
#         for h in range(num_kv):
#             for l in range(L):
#                 table_dict = self.tables[layer_id][request_id][h][l]
#                 current_hashes = hc[h, l].tolist()
#                 current_indices = idx.tolist()
#                 for i, val in enumerate(current_hashes):
#                     table_dict[val].append(current_indices[i])

#     def batch_retrieve(self, layer_id, query_hash_codes):
#         """
#         Retrieves candidates using the 2-hit filter logic found in lsh.cc.
#         Candidates must appear in at least 2 different hash tables to be considered.
#         query_hash_codes: [batch_size * num_attention_heads, L]
#         """
#         B_H, L = query_hash_codes.shape
#         query_hash_codes = query_hash_codes.cpu()

#         results = []

#         for i in range(B_H):
#             req_id = i // self.num_heads
#             local_head_id = i % self.num_heads
#             kv_head_id = local_head_id // self.num_attention_groups

#             counts = collections.defaultdict(int)

#             # Count occurrences across L tables
#             for l in range(L):
#                 val = query_hash_codes[i, l].item()
#                 bucket = self.tables[layer_id][req_id][kv_head_id][l].get(val, [])
#                 for idx in bucket:
#                     counts[idx] += 1

#             # 2-hit filter (matches lsh.cc logic)
#             candidates = [idx for idx, count in counts.items() if count >= 2]

#             if not candidates:
#                 t_cand = torch.empty(0, dtype=torch.long, device=self.device)
#             else:
#                 t_cand = torch.tensor(candidates, dtype=torch.long, device=self.device)

#             results.append(t_cand)

#         return results

# # ==========================================
# # Part 3: Sparse Attention Math
# # ==========================================

# def sparse_attention_transform(q, k, v, k_norm, K, L, head_dim, is_exact=None):
#     """
#     Implements `transform_kernel` from sparse_attention.cc.
#     CRITICAL FIX: Perform dot product and accumulation in float32.
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute scores in Float32 (Fix for precision)
#     # q: [1, D], k: [N, D] -> score: [N]
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0)

#     q_norm_f = q.float().norm(p=2)
#     k_norm_f = k_norm.float()

#     # 2. MagicPIG Probability Transform (approximate angular distance)
#     # This logic matches sparse_attention.cc/transform_kernel
#     denom = q_norm_f * k_norm_f
#     cos_theta = score_f / (denom + 1e-6)
#     # Clamp for numerical stability in acos
#     cos_theta = torch.clamp(cos_theta, -1.0 + 1e-4, 1.0 - 1e-4)

#     theta = torch.acos(cos_theta)
#     prob = 1.0 - theta / math.pi
#     p = prob.pow(K)
#     q_prob = 1.0 - p
#     w = 1.0 - q_prob.pow(L - 1) * (L * p + q_prob)

#     log_w = torch.log(w + 1e-4)

#     # 3. Apply Penalty
#     # Mask penalty for Exact tokens (Sink/Local) - they don't use LSH sampling
#     if is_exact is not None:
#         log_w = torch.where(is_exact, torch.zeros_like(log_w), log_w)

#     score_f = score_f / math.sqrt(head_dim) - log_w

#     # 4. Softmax
#     attn_probs = torch.softmax(score_f, dim=0) # [N]

#     # 5. Weighted Sum in Float32 (Fix for precision)
#     # attn_probs: [N], v: [N, D]
#     output = torch.matmul(attn_probs.unsqueeze(0), v.float()) # [1, D]

#     return output.to(v.dtype)

# def jungle_attention_transform(q, k, v, u, head_dim):
#     """
#     Self-Normalized Importance Sampling (SNIS) for Jungle Attention.
#     u: retrieval probability [N]
#     """
#     if k.shape[0] == 0:
#         return torch.zeros(1, head_dim, device=q.device, dtype=q.dtype)

#     # 1. Compute Raw Scores (Dot Product)
#     score_f = torch.matmul(q.float(), k.float().transpose(0, 1)).squeeze(0) / math.sqrt(head_dim)

#     # 2. Stable Exp
#     score_max = score_f.max()
#     score_f = score_f - score_max
#     w = torch.exp(score_f)

#     # 3. Importance Sampling Correction
#     # w_hat = w / u
#     w_hat = w / (u + 1e-6)

#     # 4. Normalized Weighted Sum
#     # Output = sum(w_hat * v) / sum(w_hat)
#     denominator = w_hat.sum() + 1e-9
#     attn_probs = w_hat / denominator

#     output = torch.matmul(attn_probs.unsqueeze(0), v.float())
#     return output.to(v.dtype)

# # ==========================================
# # Part 4: Attention Server
# # ==========================================

# class LSHSparseAttnServer:
#     """
#     Manages KV caching, LSH tables, and the hybrid attention mechanism.
#     Coordinates between Dense (Sink/Local) and Sparse (LSH) attention.
#     """
#     def __init__(self, config, K=10, L=150, batch_size=1,
#                  num_sink_tokens=4, num_local_tokens=64,
#                  max_length=8192, dense_layers=[0, 16, 32],
#                  device='cuda:0', dtype=torch.bfloat16, verbose=False,
#                  # Jungle Params
#                  use_jungle=False, jg_budget=0.05, jg_K_max=32, jg_L=128, jg_min_depth=1):

#         self.config = config
#         self.K = K
#         self.L = L
#         self.batch_size = batch_size
#         self.num_sink_tokens = num_sink_tokens
#         self.num_local_tokens = num_local_tokens
#         self.dense_layers = set(dense_layers)
#         self.device = device
#         self.dtype = dtype
#         self.verbose = verbose

#         # Jungle config
#         self.use_jungle = use_jungle
#         self.jg_budget = jg_budget
#         self.jg_K_max = jg_K_max
#         self.jg_L = jg_L
#         self.jg_min_depth = jg_min_depth

#         self.num_layers = config.num_hidden_layers
#         self.num_heads = config.num_attention_heads
#         self.num_kv_heads = config.num_key_value_heads
#         self.head_dim = config.hidden_size // self.num_heads
#         self.num_attention_groups = self.num_heads // self.num_kv_heads

#         # KV Caches
#         self.k_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]
#         self.v_cache = [torch.zeros(batch_size, self.num_kv_heads, max_length, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.current_len = [0] * batch_size

#         # Average K (centroid) for the sparse region, used for centering keys before hashing
#         self.avg_k_cache = [torch.zeros(batch_size, self.num_kv_heads, 1, self.head_dim, device=device, dtype=dtype) for _ in range(self.num_layers)]

#         self.lsh = LSH(K, L, self.num_layers, self.num_heads, self.num_kv_heads, batch_size, max_length, device)

#         # Random projection matrix for LSH (Standard)
#         self.hash_func = torch.randn((self.head_dim, K * L), device=device, dtype=dtype)
#         self.binary_pack = 2 ** torch.arange(K, device=device, dtype=torch.float32)

#         # Jungle Projections
#         if self.use_jungle:
#             print(f"🌲 Jungle Attention Enabled (L={jg_L}, K_max={jg_K_max}, Budget={jg_budget})")
#             self.jg_projs = torch.randn(self.head_dim, jg_L * jg_K_max, device=device, dtype=dtype)
#             # Store hash bits for sparse regions [layer][req_id][kv_head] -> Tensor[N_sparse, L, K_max]
#             self.jg_hash_cache = collections.defaultdict(dict)

#         # Track the boundary of the sparse region per request
#         self.sparse_boundaries = {}

#     def clear(self):
#         self.lsh.clear()
#         for i in range(self.batch_size):
#             self.current_len[i] = 0
#         for l in range(self.num_layers):
#             self.k_cache[l].zero_()
#             self.v_cache[l].zero_()
#             self.avg_k_cache[l].zero_()
#         self.sparse_boundaries = {}
#         if self.use_jungle:
#             self.jg_hash_cache.clear()

#     def step(self):
#         """
#         Advances the sequence length counter for all requests.
#         Must be called ONCE after all layers have processed the current token.
#         """
#         for i in range(self.batch_size):
#             self.current_len[i] += 1

#     def fill(self, layer_idx, request_id, key_states, value_states, start_pos):
#         """
#         Updates KV cache and fills LSH tables for the 'offload' region (Sparse).
#         Corresponds to attnserver.py fill logic.
#         """
#         seq_len = key_states.shape[0]
#         end_pos = start_pos + seq_len

#         # 1. Update KV Cache (Store RAW keys)
#         self.k_cache[layer_idx][request_id, :, start_pos:end_pos, :] = key_states.transpose(0, 1)
#         self.v_cache[layer_idx][request_id, :, start_pos:end_pos, :] = value_states.transpose(0, 1)

#         # Update current length during prefill
#         if end_pos > self.current_len[request_id]:
#             self.current_len[request_id] = end_pos

#         # 2. Update LSH Tables (only for sparse layers during Pre-fill)
#         if layer_idx not in self.dense_layers:
#             # Logic matches attnserver.py: index tokens between [sink, end-local]
#             idx_start = max(start_pos, self.num_sink_tokens)
#             idx_end = end_pos - self.num_local_tokens

#             # Store boundary: tokens before this are Sparse (hashed), after are Local (Exact)
#             self.sparse_boundaries[request_id] = max(self.num_sink_tokens, idx_end)

#             # Only index if there is a sparse region
#             if idx_end > idx_start:
#                 keys_to_index = self.k_cache[layer_idx][request_id, :, idx_start:idx_end, :]

#                 # Center keys using mean of the sparse region (float32 for precision)
#                 avg_k = keys_to_index.float().mean(dim=1, keepdim=True).to(self.dtype)
#                 self.avg_k_cache[layer_idx][request_id] = avg_k

#                 centered_keys = keys_to_index - avg_k

#                 if self.use_jungle:
#                     # --- Jungle Hashing ---
#                     # Compute L*K_max bits for Jungle Forest
#                     # [num_kv, N, D] @ [D, L*K_max] -> [num_kv, N, L*K_max]
#                     jg_proj = torch.matmul(centered_keys, self.jg_projs)
#                     jg_bits = (jg_proj > 0).float() # [num_kv, N, L*K_max]

#                     # Reshape to [num_kv, N, L, K_max]
#                     n_sparse = jg_bits.shape[1]
#                     jg_bits = jg_bits.view(self.num_kv_heads, n_sparse, self.jg_L, self.jg_K_max)

#                     # Store in cache
#                     if layer_idx not in self.jg_hash_cache: self.jg_hash_cache[layer_idx] = {}
#                     self.jg_hash_cache[layer_idx][request_id] = jg_bits
#                 else:
#                     # --- Standard LSH ---
#                     # Compute LSH hashes
#                     projected = torch.matmul(centered_keys, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_kv_heads, -1, self.L, self.K)
#                     buckets = torch.matmul(bits, self.binary_pack).long()
#                     buckets = buckets.permute(0, 2, 1)

#                     indices = torch.arange(idx_start, idx_end, device=self.device)
#                     self.lsh.fill(layer_idx, request_id, buckets, indices)

#     def decode(self, query_states, key_states, value_states, layer_idx):
#         """
#         Performs decoding step with DEBUG STATEMENTS added.
#         """
#         bsz, n_heads, q_len, dim = query_states.shape
#         assert q_len == 1, "Sparse decode expects query length 1"

#         # --- Update Cache with new token ---
#         for req_id in range(bsz):
#             curr_len = self.current_len[req_id]
#             k_slice = key_states[req_id]
#             v_slice = value_states[req_id]

#             self.k_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = k_slice
#             self.v_cache[layer_idx][req_id, :, curr_len:curr_len+1, :] = v_slice

#         hidden_states_list = []

#         for req_id in range(bsz):
#             q_heads = query_states[req_id, :, 0, :]
#             curr_len = self.current_len[req_id]

#             if layer_idx in self.dense_layers:
#                 # ... (Dense logic remains the same) ...
#                 k = self.k_cache[layer_idx][req_id, :, :curr_len, :]
#                 v = self.v_cache[layer_idx][req_id, :, :curr_len, :]
#                 k = repeat_kv(k.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)
#                 v = repeat_kv(v.unsqueeze(0), self.num_heads // self.num_kv_heads).squeeze(0)

#                 scores = torch.matmul(q_heads.float().unsqueeze(1), k.float().transpose(1, 2)) / math.sqrt(dim)
#                 attn = torch.softmax(scores, dim=-1)
#                 out = torch.matmul(attn, v.float()).squeeze(1)
#                 hidden_states_list.append(out.to(self.dtype))

#             else:
#                 # --- Sparse Retrieval Logic ---
#                 head_outputs = []

#                 # Pre-compute Q hash for Jungle if enabled
#                 if self.use_jungle:
#                     norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)
#                     # [Heads, D] @ [D, L*K] -> [Heads, L*K]
#                     jg_q_proj = torch.matmul(norm_q, self.jg_projs)
#                     jg_q_bits = (jg_q_proj > 0).float().view(self.num_heads, self.jg_L, self.jg_K_max)
#                 else:
#                     norm_q = q_heads / (q_heads.norm(p=2, dim=-1, keepdim=True) + 1e-6)
#                     projected = torch.matmul(norm_q, self.hash_func)
#                     bits = (projected > 0).float()
#                     bits = bits.view(self.num_heads, self.L, self.K)
#                     q_buckets = torch.matmul(bits, self.binary_pack).long()
#                     idx_list = self.lsh.batch_retrieve(layer_idx, q_buckets.unsqueeze(0).view(-1, self.L))

#                 for h in range(self.num_heads):
#                     kv_head = h // self.num_attention_groups
#                     sparse_boundary = self.sparse_boundaries.get(req_id, 0)

#                     # 1. Sink Tokens
#                     sink_indices = torch.arange(0, min(curr_len, self.num_sink_tokens), device=self.device)

#                     # 2. Local Tokens
#                     if curr_len > sparse_boundary:
#                         local_start = max(sparse_boundary, 0)
#                         local_indices = torch.arange(local_start, curr_len, device=self.device)
#                     else:
#                         local_indices = torch.empty(0, dtype=torch.long, device=self.device)

#                     # 3. Sparse Tokens
#                     sparse_indices = torch.empty(0, dtype=torch.long, device=self.device)
#                     sparse_u = torch.empty(0, dtype=torch.float32, device=self.device)

#                     if self.use_jungle:
#                         # --- Jungle Retrieval ---
#                         # Check if we have sparse keys for this request/layer
#                         if (layer_idx in self.jg_hash_cache and
#                             req_id in self.jg_hash_cache[layer_idx]):

#                             # Get cached K bits: [N_sparse, L, K_max]
#                             # Note: self.jg_hash_cache stores [num_kv_heads, N, L, K]
#                             k_bits = self.jg_hash_cache[layer_idx][req_id][kv_head]
#                             q_bits_h = jg_q_bits[h] # [L, K_max]

#                             # Compute prefix matches (depths)
#                             # (q == k) -> [N, L, K]
#                             # cumprod to find prefix length
#                             match = (k_bits == q_bits_h.unsqueeze(0)).int()
#                             # cumprod over K dim, sum over K dim -> depth
#                             depths = match.cumprod(dim=-1).sum(dim=-1) # [N, L]

#                             # Max depth across L trees
#                             max_d, _ = depths.max(dim=-1) # [N]

#                             # Select Top-B based on max_d
#                             N_sparse = max_d.shape[0]
#                             budget = int(N_sparse * self.jg_budget)
#                             if budget > 0:
#                                 sorted_d, sorted_idx = torch.sort(max_d, descending=True)
#                                 # Apply min depth filter
#                                 valid_mask = sorted_d >= self.jg_min_depth
#                                 valid_idx = sorted_idx[valid_mask]
#                                 take = min(budget, valid_idx.numel())

#                                 if take > 0:
#                                     # sparse_indices_raw are relative to sparse region start
#                                     # Sparse region starts at num_sink_tokens
#                                     rel_indices = valid_idx[:take]
#                                     sparse_indices = rel_indices + self.num_sink_tokens

#                                     # Compute U weights
#                                     # d_thresh is the depth of the worst selected item (approximation)
#                                     d_thresh = max_d[rel_indices].min().float()

#                                     # Need Cosine Sim for p
#                                     # Re-compute dot product for selected keys?
#                                     # Approximating p from depth is hard without calibration.
#                                     # Jungle paper suggests estimating p based on d.
#                                     # Or we compute exact dot product for selected keys to get p.
#                                     # Let's compute exact p for the selected set to be precise for SNIS.
#                                     k_sel_sparse = self.k_cache[layer_idx][req_id, kv_head, sparse_indices, :]
#                                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]

#                                     # Normalize centered
#                                     Q_norm_h = norm_q[h]
#                                     K_centered_norm = F.normalize(k_sel_sparse - avg_k, dim=-1)
#                                     dot = (Q_norm_h @ K_centered_norm.T).clamp(-0.999, 0.999)
#                                     p_est = 1.0 - (torch.acos(dot) / math.pi)

#                                     p_d = p_est.pow(d_thresh)
#                                     sparse_u = (1.0 - (1.0 - p_d).pow(self.jg_L)).clamp(1e-6, 1.0)

#                     else:
#                         # --- Standard LSH Retrieval ---
#                         sparse_indices_raw = idx_list[h]
#                         if sparse_indices_raw.numel() > 0:
#                             sparse_indices = sparse_indices_raw[(sparse_indices_raw >= self.num_sink_tokens) & (sparse_indices_raw < sparse_boundary)]
#                             # Standard LSH weights imply u depends on collision prob, usually calculated inside transform or assumed
#                             # Here we just pass ones for sparse_u later if needed, but standard LSH code
#                             # usually applies a different correction. The helper `sparse_attention_transform`
#                             # computes the magicpig probability internally.

#                     # Combine indices
#                     # full_indices: [Sink..., Sparse..., Local...]
#                     full_indices = torch.cat([sink_indices, sparse_indices, local_indices]).unique()

#                     # ========================================================
#                     # DEBUG START: Sparsity Visualization
#                     # ========================================================
#                     if self.verbose and h == 0 and req_id == 0:
#                         total_tokens = curr_len
#                         kept = full_indices.numel()

#                         if total_tokens > (self.num_sink_tokens + self.num_local_tokens):
#                             ignored = total_tokens - kept
#                             sparsity_ratio = (ignored / total_tokens) * 100

#                             method_name = "Jungle" if self.use_jungle else "MagicPIG"
#                             print(f"\n[DEBUG] {method_name} | Layer {layer_idx} | SeqLen {total_tokens}")
#                             print(f"  Status:  Attending to {kept} tokens (Ignored {ignored})")
#                             print(f"  Sparsity: {sparsity_ratio:.2f}%")

#                             if self.use_jungle and sparse_indices.numel() > 0:
#                                 print(f"  Jungle:  Selected {sparse_indices.numel()} sparse tokens")
#                                 print(f"           Min Weight u: {sparse_u.min().item():.4f}")
#                     # ========================================================

#                     # Prepare for attention transform
#                     k_sel = self.k_cache[layer_idx][req_id, kv_head, full_indices, :]
#                     v_sel = self.v_cache[layer_idx][req_id, kv_head, full_indices, :]

#                     avg_k = self.avg_k_cache[layer_idx][req_id, kv_head, 0, :]
#                     k_sel_centered = k_sel - avg_k

#                     q_h = q_heads[h].unsqueeze(0)

#                     if self.use_jungle:
#                         # Construct full U vector
#                         # Sink/Local have u=1.0
#                         # Sparse have calculated u
#                         # We need to map sparse_u back to full_indices positions
#                         # This is tricky with unique(). Simplification:
#                         # Re-construct indices without unique to align u
#                         # (Assume disjoint sets for simplicity or just assign 1 to non-sparse)

#                         # Create a map for u values
#                         u_map = torch.ones(curr_len, device=self.device, dtype=torch.float32)
#                         if sparse_indices.numel() > 0:
#                             u_map[sparse_indices] = sparse_u

#                         u_vec = u_map[full_indices]

#                         out_h = jungle_attention_transform(
#                             q_h, k_sel_centered, v_sel, u_vec, self.head_dim
#                         )
#                     else:
#                         is_exact = (full_indices < self.num_sink_tokens) | (full_indices >= sparse_boundary)
#                         k_norm_sel = k_sel_centered.float().norm(p=2, dim=-1)
#                         out_h = sparse_attention_transform(
#                             q_h, k_sel_centered, v_sel, k_norm_sel,
#                             self.K, self.L, self.head_dim,
#                             is_exact=is_exact
#                         )

#                     head_outputs.append(out_h)

#                 hidden_states_list.append(torch.cat(head_outputs, dim=0))

#         final_out = torch.stack(hidden_states_list, dim=0).view(bsz, 1, n_heads * dim)
#         return final_out

# # ==========================================
# # Part 5: Model Wrappers
# # ==========================================

# class LLMLayer:
#     """
#     Wraps a HuggingFace LlamaDecoderLayer.
#     Matches the structure of models/llama.py.
#     """
#     def __init__(self, layer_idx, hf_layer, device):
#         self.layer_idx = layer_idx
#         self.device = device

#         self.wq = hf_layer.self_attn.q_proj.weight.detach().to(device)
#         self.wk = hf_layer.self_attn.k_proj.weight.detach().to(device)
#         self.wv = hf_layer.self_attn.v_proj.weight.detach().to(device)
#         self.wo = hf_layer.self_attn.o_proj.weight.detach().to(device)

#         self.gate_proj = hf_layer.mlp.gate_proj.weight.detach().to(device)
#         self.up_proj = hf_layer.mlp.up_proj.weight.detach().to(device)
#         self.down_proj = hf_layer.mlp.down_proj.weight.detach().to(device)

#         self.input_layernorm_weight = hf_layer.input_layernorm.weight.detach().to(device)
#         self.input_layernorm_eps = hf_layer.input_layernorm.variance_epsilon
#         self.post_attention_layernorm_weight = hf_layer.post_attention_layernorm.weight.detach().to(device)
#         self.post_attention_layernorm_eps = hf_layer.post_attention_layernorm.variance_epsilon

#     def forward(self, hidden_states, position_ids, attn_server, cos_cache, sin_cache, is_prefill=False):
#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.input_layernorm_weight, self.input_layernorm_eps)

#         bsz, q_len, _ = hidden_states.shape
#         q = F.linear(hidden_states, self.wq)
#         k = F.linear(hidden_states, self.wk)
#         v = F.linear(hidden_states, self.wv)

#         n_heads = attn_server.num_heads
#         n_kv_heads = attn_server.num_kv_heads
#         head_dim = attn_server.head_dim

#         q = q.view(bsz, q_len, n_heads, head_dim).transpose(1, 2)
#         k = k.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)
#         v = v.view(bsz, q_len, n_kv_heads, head_dim).transpose(1, 2)

#         q = apply_rotary_pos_emb(q, cos_cache, sin_cache, position_ids)
#         k = apply_rotary_pos_emb(k, cos_cache, sin_cache, position_ids)

#         if is_prefill:
#             # Prefill: Use Dense Attention & Build LSH Index
#             for i in range(bsz):
#                 # Call fill to populate LSH with the Prompt's sparse region
#                 attn_server.fill(self.layer_idx, i, k[i].permute(1, 0, 2), v[i].permute(1, 0, 2), start_pos=position_ids[i, 0].item())

#             k_rep = repeat_kv(k, n_heads // n_kv_heads)
#             v_rep = repeat_kv(v, n_heads // n_kv_heads)
#             # Use PyTorch SDPA for dense prefill (robust implementation)
#             attn_output = F.scaled_dot_product_attention(q, k_rep, v_rep, attn_mask=None, is_causal=True)
#             attn_output = attn_output.transpose(1, 2).reshape(bsz, q_len, -1)
#         else:
#             # Decode: Use Sparse Attention
#             # Passes current token k, v to decode for local caching
#             attn_output = attn_server.decode(q, k, v, self.layer_idx)

#         hidden_states = F.linear(attn_output, self.wo)
#         hidden_states = residual + hidden_states

#         residual = hidden_states
#         hidden_states = manual_rmsnorm(hidden_states, self.post_attention_layernorm_weight, self.post_attention_layernorm_eps)
#         up = F.linear(hidden_states, self.up_proj)
#         gate = F.linear(hidden_states, self.gate_proj)
#         down = F.linear(F.silu(gate) * up, self.down_proj)
#         hidden_states = residual + down

#         return hidden_states

# class LLM:
#     """
#     Main Model Class.
#     Matches models/llama.py structure.
#     """
#     def __init__(self, model_name, K=10, L=150, max_length=2048, device='cuda:0', use_jungle=False):
#         self.device = device
#         self.config = LlamaConfig.from_pretrained(model_name)
#         self.max_length = max_length

#         print(f"Loading Model: {model_name}...")
#         hf_model = LlamaForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

#         self.embed_tokens = hf_model.model.embed_tokens.weight.detach().to(device)
#         self.lm_head = hf_model.lm_head.weight.detach().to(device)
#         self.norm_weight = hf_model.model.norm.weight.detach().to(device)
#         self.norm_eps = hf_model.model.norm.variance_epsilon

#         self.inv_freq = hf_model.model.rotary_emb.inv_freq.detach().to(device)
#         t = torch.arange(max_length, device=device, dtype=self.inv_freq.dtype)
#         freqs = torch.outer(t, self.inv_freq)
#         emb = torch.cat((freqs, freqs), dim=-1)
#         self.cos_cache = emb.cos().to(torch.bfloat16)
#         self.sin_cache = emb.sin().to(torch.bfloat16)

#         self.layers = []
#         for idx, layer in enumerate(hf_model.model.layers):
#             self.layers.append(LLMLayer(idx, layer, device))
#             hf_model.model.layers[idx] = None
#             gc.collect()

#         self.attn_server = LSHSparseAttnServer(self.config, K=K, L=L, max_length=max_length, device=device, use_jungle=use_jungle)

#     def generate(self, input_ids, max_tokens=100, temperature=0.6, verbose=False):
#         self.attn_server.verbose = verbose
#         bsz, seq_len = input_ids.shape
#         position_ids = torch.arange(seq_len, device=self.device).unsqueeze(0)

#         print("Prefilling (Dense Attention)...")
#         t0 = time.time()
#         hidden_states = F.embedding(input_ids, self.embed_tokens)

#         # Prefill Phase
#         for layer in self.layers:
#             hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=True)

#         generated = []
#         curr_pos = seq_len

#         logits = F.linear(manual_rmsnorm(hidden_states[:,-1:], self.norm_weight, self.norm_eps), self.lm_head)
#         next_token = topp_temperature_decode(logits, temperature)
#         generated.append(next_token.item())
#         t1 = time.time()
#         print(f"Prefill done in {t1-t0:.2f}s")

#         print("Generating (Sparse Attention)...")
#         # Decode Phase
#         for i in range(max_tokens):
#             if verbose: print(f"--- Step {i} ---")
#             input_ids = next_token # [bsz, 1]
#             position_ids = torch.tensor([[curr_pos]], device=self.device)

#             hidden_states = F.embedding(input_ids, self.embed_tokens)

#             for layer in self.layers:
#                 hidden_states = layer.forward(hidden_states, position_ids, self.attn_server, self.cos_cache, self.sin_cache, is_prefill=False)

#             # --- CRITICAL FIX: Update Sequence Length ONCE per step ---
#             self.attn_server.step()

#             logits = F.linear(manual_rmsnorm(hidden_states, self.norm_weight, self.norm_eps), self.lm_head)
#             next_token = topp_temperature_decode(logits, temperature)
#             generated.append(next_token.item())
#             curr_pos += 1

#             # Stop tokens for Llama 3
#             if next_token.item() in [128001, 128009]:
#                 break

#         return generated

# # ==========================================
# # Main Execution
# # ==========================================

# if __name__ == "__main__":
#     # Example Usage
#     MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
#     DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

#     try:
#         # Initialize model with Jungle Attention Enabled
#         # Set use_jungle=True to test the new logic
#         llm = LLM(MODEL_NAME, K=10, L=150, max_length=1024, device=DEVICE, use_jungle=False)

#         tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#         prompt = "Answer very concisely. In the rapidly evolving field of elementary mathematics, teachers are always looking for new ways to help students work efficiently with numbers. One useful idea involves focusing only on the most important values in a problem, which can make calculations quicker and easier. The SimpleSUM method is a good example of this approach, grouping numbers with similar sizes so that students can estimate results without checking every single value. This technique can greatly improve how learners handle long lists of numbers in everyday situations. If a list contains 20 numbers and a student keeps only the 5 largest ones to make an estimate, and those 5 numbers are 8, 9, 10, 11, and 12, what is the student’s estimated total?"


#         print(f"\nPrompt: {prompt}")
#         input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)

#         # Generate with Verbose Debugging
#         start = time.time()
#         output_ids = llm.generate(input_ids, max_tokens=20, temperature=0.7, verbose=False)
#         end = time.time()

#         decoded = tokenizer.decode(output_ids)
#         print(f"\nGenerated: {decoded}")
#         print(f"Generation Time: {end - start:.2f}s")

#     except Exception as e:
#         print(f"\nError encountered: {e}")